<a href="https://colab.research.google.com/github/yashvip-beep/GST-FINDERRR/blob/main/GST_TAXORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit pandas pytest reportlab pillow pyzbar
!apt-get install -y -q libzbar0 > /dev/null 2>&1

In [ ]:
import os, shutil
shutil.rmtree("/content/gst-rate-finder", ignore_errors=True)
os.makedirs("/content/gst-rate-finder/data", exist_ok=True)
os.makedirs("/content/gst-rate-finder/tests", exist_ok=True)
!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared.deb > /dev/null
!cloudflared --version

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
cloudflared version 2026.11.1 (built 2026-11-17-15:04 UTC)


In [ ]:
%%writefile /content/gst-rate-finder/requirements.txt
streamlit>=1.37
pandas>=2.0
pytest>=8.0
reportlab>=4.0
pillow>=10.0
pyzbar>=0.1.9

Writing /content/gst-rate-finder/requirements.txt


In [ ]:
%%writefile /content/gst-rate-finder/gst_engine.py
"""Core GST logic. No Streamlit imports, so it can be unit-tested with pytest.
Rates are read from data/gst_rates.json, never hard-coded."""
from __future__ import annotations

import csv
import difflib
import io
import json
import re
from decimal import ROUND_HALF_UP, Decimal, InvalidOperation
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
DATA_FILE = BASE_DIR / "data" / "gst_rates.json"

NOT_FOUND_MSG = "Item not found — please check spelling or try a related term."
DISCLAIMER = (
    "This tool is provided for general information and educational purposes. GST classification "
    "and tax treatment depend on the precise nature of the goods or services, facts of the "
    "transaction, applicable notifications and conditions. Verify the relevant official notification "
    "and obtain advice from a qualified tax professional before relying on the result for invoicing, "
    "return filing or other compliance. The tool does not constitute legal or tax advice."
)

HUNDRED = Decimal("100")
CENT = Decimal("0.01")
ZERO = Decimal("0")
FUZZY_CUTOFF = 0.78

REQUIRED_FIELDS = {
    "id", "kind", "name", "aliases", "hsn_sac", "category", "treatment", "cgst_pct",
    "notification", "serial", "condition", "effective_from", "source_url", "note",
}
VALID_KINDS = {"goods", "service"}
VALID_TREATMENTS = {"taxable", "exempt"}


def _validate(records: list[dict]) -> None:
    seen: set[str] = set()
    for rec in records:
        missing = REQUIRED_FIELDS - rec.keys()
        if missing:
            raise ValueError(f"Record {rec.get('id', '?')} is missing fields: {sorted(missing)}")
        if rec["id"] in seen:
            raise ValueError(f"Duplicate record id: {rec['id']}")
        seen.add(rec["id"])
        if rec["kind"] not in VALID_KINDS or rec["treatment"] not in VALID_TREATMENTS:
            raise ValueError(f"Record {rec['id']} has an invalid kind or treatment")
        rate = Decimal(rec["cgst_pct"])
        if rec["treatment"] == "exempt" and rate != ZERO:
            raise ValueError(f"Exempt record {rec['id']} must have cgst_pct of 0")


def load_dataset(path: Path = DATA_FILE) -> tuple[dict, list[dict]]:
    with open(path, encoding="utf-8") as fh:
        data = json.load(fh)
    records = data["records"]
    _validate(records)
    return data["meta"], records


def normalise(text: str) -> str:
    text = re.sub(r"[^a-z0-9 ]+", " ", (text or "").lower())
    return re.sub(r"\s+", " ", text).strip()


def digits_only(text: str) -> str:
    return re.sub(r"\D", "", text or "")


def hsn_codes(rec: dict) -> list[str]:
    parts = re.split(r"[/,]", rec["hsn_sac"])
    return [digits_only(p) for p in parts if digits_only(p)]


def _word_in(needle: str, haystack: str) -> bool:
    return re.search(rf"(?<![a-z0-9]){re.escape(needle)}(?![a-z0-9])", haystack) is not None


def central_pct(rec: dict) -> Decimal:
    return Decimal(rec["cgst_pct"]) if rec["treatment"] == "taxable" else ZERO


def total_gst_pct(rec: dict) -> Decimal:
    return central_pct(rec) * 2


def rate_split(rec: dict) -> dict[str, Decimal]:
    c = central_pct(rec)
    return {"cgst": c, "sgst": c, "igst": c * 2}


def fmt_pct(value: Decimal) -> str:
    return f"{format(value.normalize(), 'f')}%"


def status_text(rec: dict) -> str:
    if rec["treatment"] == "taxable":
        return "GST APPLICABLE"
    return "GST NOT APPLICABLE (Exempt, nil rate)"


def search_text(query: str, records: list[dict], kind: str) -> tuple[list[dict], list[dict]]:
    """Return (exact_matches, suggestions). Suggestions are fuzzy and NOT confirmed."""
    q = normalise(query)
    if not q:
        return [], []
    pool = [r for r in records if r["kind"] == kind]
    exact = []
    for rec in pool:
        terms = [normalise(rec["name"])] + [normalise(a) for a in rec["aliases"]]
        if any(_word_in(q, term) for term in terms):
            exact.append(rec)

    exact_ids = {r["id"] for r in exact}
    vocab: dict[str, list[dict]] = {}
    for rec in pool:
        if rec["id"] in exact_ids:
            continue
        for term in [normalise(rec["name"])] + [normalise(a) for a in rec["aliases"]]:
            vocab.setdefault(term, []).append(rec)

    suggestions: list[dict] = []
    seen: set[str] = set()
    for match in difflib.get_close_matches(q, list(vocab), n=8, cutoff=FUZZY_CUTOFF):
        for rec in vocab[match]:
            if rec["id"] not in seen:
                seen.add(rec["id"])
                suggestions.append(rec)
    return exact, suggestions


def search_code(code: str, records: list[dict], kind: str) -> list[dict]:
    digits = digits_only(code)
    if len(digits) not in (4, 6, 8):
        raise ValueError("Enter a 4, 6 or 8-digit HSN/SAC code.")
    return [r for r in records
            if r["kind"] == kind and any(c.startswith(digits) for c in hsn_codes(r))]


def _to_decimal(value, label: str) -> Decimal:
    try:
        number = Decimal(str(value))
    except (InvalidOperation, ValueError):
        raise ValueError(f"{label} must be a number.") from None
    if not number.is_finite():
        raise ValueError(f"{label} must be a finite number.")
    return number


def _q(value: Decimal) -> Decimal:
    return value.quantize(CENT, rounding=ROUND_HALF_UP)


def calculate(amount, rate_pct, supply: str, mode: str) -> dict[str, str]:
    base = _to_decimal(amount, "Amount")
    rate = _to_decimal(rate_pct, "GST rate")
    if base <= 0:
        raise ValueError("Amount must be greater than zero.")
    if rate < 0 or rate > HUNDRED:
        raise ValueError("GST rate must be between 0% and 100%.")
    if supply not in ("intra", "inter"):
        raise ValueError("Supply type must be 'intra' or 'inter'.")

    if mode == "add":
        taxable = _q(base)
        tax = _q(taxable * rate / HUNDRED)
        final = taxable + tax
    elif mode == "extract":
        taxable = _q(base * HUNDRED / (HUNDRED + rate))
        final = _q(base)
        tax = final - taxable
    else:
        raise ValueError("Mode must be 'add' or 'extract'.")

    if supply == "intra":
        cgst = _q(tax / 2)
        sgst = tax - cgst
        igst = ZERO
    else:
        cgst = ZERO
        sgst = ZERO
        igst = tax

    return {
        "taxable_value": f"{taxable:.2f}",
        "cgst": f"{cgst:.2f}",
        "sgst_utgst": f"{sgst:.2f}",
        "igst": f"{igst:.2f}",
        "total_tax": f"{tax:.2f}",
        "final_amount": f"{final:.2f}",
        "rounding_note": (
            "Amounts are rounded to 2 decimal places (half-up). In intra-State mode, CGST is half "
            "the tax and SGST takes any paise remainder."
        ),
    }


def result_row(rec: dict, supply: str, meta: dict, downloaded_on: str) -> dict[str, str]:
    split = rate_split(rec)
    intra = supply == "intra"
    return {
        "item": rec["name"],
        "type": rec["kind"],
        "hsn_sac": rec["hsn_sac"],
        "status": status_text(rec),
        "total_gst": fmt_pct(total_gst_pct(rec)),
        "supply": "Intra-State" if intra else "Inter-State",
        "cgst": fmt_pct(split["cgst"]) if intra else "0%",
        "sgst_utgst": fmt_pct(split["sgst"]) if intra else "0%",
        "igst": "0%" if intra else fmt_pct(split["igst"]),
        "condition": rec["condition"],
        "notification": rec["notification"],
        "entry": rec["serial"],
        "effective_from": rec["effective_from"],
        "source_url": rec["source_url"],
        "dataset_version": meta["dataset_version"],
        "downloaded_on": downloaded_on,
        "disclaimer": DISCLAIMER,
    }


def to_csv(rows: list[dict]) -> bytes:
    if not rows:
        return b""
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue().encode("utf-8-sig")

Writing /content/gst-rate-finder/gst_engine.py


In [ ]:
%%writefile /content/gst-rate-finder/gst_engine.py
"""Core GST logic. No Streamlit imports, so it can be unit-tested with pytest.
Rates are read from data/gst_rates.json, never hard-coded."""
from __future__ import annotations

import csv
import difflib
import io
import json
import re
from decimal import ROUND_HALF_UP, Decimal, InvalidOperation
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
DATA_FILE = BASE_DIR / "data" / "gst_rates.json"

NOT_FOUND_MSG = "Item not found — please check spelling or try a related term."
DISCLAIMER = (
    "This tool is provided for general information and educational purposes. GST classification "
    "and tax treatment depend on the precise nature of the goods or services, facts of the "
    "transaction, applicable notifications and conditions. Verify the relevant official notification "
    "and obtain advice from a qualified tax professional before relying on the result for invoicing, "
    "return filing or other compliance. The tool does not constitute legal or tax advice."
)

HUNDRED = Decimal("100")
CENT = Decimal("0.01")
ZERO = Decimal("0")
FUZZY_CUTOFF = 0.78

REQUIRED_FIELDS = {
    "id", "kind", "name", "aliases", "hsn_sac", "category", "treatment", "cgst_pct",
    "notification", "serial", "condition", "effective_from", "source_url", "note",
}
VALID_KINDS = {"goods", "service"}
VALID_TREATMENTS = {"taxable", "exempt"}


def _validate(records: list[dict]) -> None:
    seen: set[str] = set()
    for rec in records:
        missing = REQUIRED_FIELDS - rec.keys()
        if missing:
            raise ValueError(f"Record {rec.get('id', '?')} is missing fields: {sorted(missing)}")
        if rec["id"] in seen:
            raise ValueError(f"Duplicate record id: {rec['id']}")
        seen.add(rec["id"])
        if rec["kind"] not in VALID_KINDS or rec["treatment"] not in VALID_TREATMENTS:
            raise ValueError(f"Record {rec['id']} has an invalid kind or treatment")
        rate = Decimal(rec["cgst_pct"])
        if rec["treatment"] == "exempt" and rate != ZERO:
            raise ValueError(f"Exempt record {rec['id']} must have cgst_pct of 0")


def load_dataset(path: Path = DATA_FILE) -> tuple[dict, list[dict]]:
    with open(path, encoding="utf-8") as fh:
        data = json.load(fh)
    records = data["records"]
    _validate(records)
    return data["meta"], records


def normalise(text: str) -> str:
    text = re.sub(r"[^a-z0-9 ]+", " ", (text or "").lower())
    return re.sub(r"\s+", " ", text).strip()


def digits_only(text: str) -> str:
    return re.sub(r"\D", "", text or "")


def hsn_codes(rec: dict) -> list[str]:
    parts = re.split(r"[/,]", rec["hsn_sac"])
    return [digits_only(p) for p in parts if digits_only(p)]


def _word_in(needle: str, haystack: str) -> bool:
    return re.search(rf"(?<![a-z0-9]){re.escape(needle)}(?![a-z0-9])", haystack) is not None


def central_pct(rec: dict) -> Decimal:
    return Decimal(rec["cgst_pct"]) if rec["treatment"] == "taxable" else ZERO


def total_gst_pct(rec: dict) -> Decimal:
    return central_pct(rec) * 2


def rate_split(rec: dict) -> dict[str, Decimal]:
    c = central_pct(rec)
    return {"cgst": c, "sgst": c, "igst": c * 2}


def fmt_pct(value: Decimal) -> str:
    return f"{format(value.normalize(), 'f')}%"


def status_text(rec: dict) -> str:
    if rec["treatment"] == "taxable":
        return "GST APPLICABLE"
    return "GST NOT APPLICABLE (Exempt, nil rate)"


def search_text(query: str, records: list[dict], kind: str) -> tuple[list[dict], list[dict]]:
    """Return (exact_matches, suggestions). Suggestions are fuzzy and NOT confirmed."""
    q = normalise(query)
    if not q:
        return [], []
    pool = [r for r in records if r["kind"] == kind]
    exact = []
    for rec in pool:
        terms = [normalise(rec["name"])] + [normalise(a) for a in rec["aliases"]]
        if any(_word_in(q, term) for term in terms):
            exact.append(rec)

    exact_ids = {r["id"] for r in exact}
    vocab: dict[str, list[dict]] = {}
    for rec in pool:
        if rec["id"] in exact_ids:
            continue
        for term in [normalise(rec["name"])] + [normalise(a) for a in rec["aliases"]]:
            vocab.setdefault(term, []).append(rec)

    suggestions: list[dict] = []
    seen: set[str] = set()
    for match in difflib.get_close_matches(q, list(vocab), n=8, cutoff=FUZZY_CUTOFF):
        for rec in vocab[match]:
            if rec["id"] not in seen:
                seen.add(rec["id"])
                suggestions.append(rec)
    return exact, suggestions


def search_code(code: str, records: list[dict], kind: str) -> list[dict]:
    digits = digits_only(code)
    if len(digits) not in (4, 6, 8):
        raise ValueError("Enter a 4, 6 or 8-digit HSN/SAC code.")
    return [r for r in records
            if r["kind"] == kind and any(c.startswith(digits) for c in hsn_codes(r))]


def _to_decimal(value, label: str) -> Decimal:
    try:
        number = Decimal(str(value))
    except (InvalidOperation, ValueError):
        raise ValueError(f"{label} must be a number.") from None
    if not number.is_finite():
        raise ValueError(f"{label} must be a finite number.")
    return number


def _q(value: Decimal) -> Decimal:
    return value.quantize(CENT, rounding=ROUND_HALF_UP)


def calculate(amount, rate_pct, supply: str, mode: str) -> dict[str, str]:
    base = _to_decimal(amount, "Amount")
    rate = _to_decimal(rate_pct, "GST rate")
    if base <= 0:
        raise ValueError("Amount must be greater than zero.")
    if rate < 0 or rate > HUNDRED:
        raise ValueError("GST rate must be between 0% and 100%.")
    if supply not in ("intra", "inter"):
        raise ValueError("Supply type must be 'intra' or 'inter'.")

    if mode == "add":
        taxable = _q(base)
        tax = _q(taxable * rate / HUNDRED)
        final = taxable + tax
    elif mode == "extract":
        taxable = _q(base * HUNDRED / (HUNDRED + rate))
        final = _q(base)
        tax = final - taxable
    else:
        raise ValueError("Mode must be 'add' or 'extract'.")

    if supply == "intra":
        cgst = _q(tax / 2)
        sgst = tax - cgst
        igst = ZERO
    else:
        cgst = ZERO
        sgst = ZERO
        igst = tax

    return {
        "taxable_value": f"{taxable:.2f}",
        "cgst": f"{cgst:.2f}",
        "sgst_utgst": f"{sgst:.2f}",
        "igst": f"{igst:.2f}",
        "total_tax": f"{tax:.2f}",
        "final_amount": f"{final:.2f}",
        "rounding_note": (
            "Amounts are rounded to 2 decimal places (half-up). In intra-State mode, CGST is half "
            "the tax and SGST takes any paise remainder."
        ),
    }


def result_row(rec: dict, supply: str, meta: dict, downloaded_on: str) -> dict[str, str]:
    split = rate_split(rec)
    intra = supply == "intra"
    return {
        "item": rec["name"],
        "type": rec["kind"],
        "hsn_sac": rec["hsn_sac"],
        "status": status_text(rec),
        "total_gst": fmt_pct(total_gst_pct(rec)),
        "supply": "Intra-State" if intra else "Inter-State",
        "cgst": fmt_pct(split["cgst"]) if intra else "0%",
        "sgst_utgst": fmt_pct(split["sgst"]) if intra else "0%",
        "igst": "0%" if intra else fmt_pct(split["igst"]),
        "condition": rec["condition"],
        "notification": rec["notification"],
        "entry": rec["serial"],
        "effective_from": rec["effective_from"],
        "source_url": rec["source_url"],
        "dataset_version": meta["dataset_version"],
        "downloaded_on": downloaded_on,
        "disclaimer": DISCLAIMER,
    }


def to_csv(rows: list[dict]) -> bytes:
    if not rows:
        return b""
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue().encode("utf-8-sig")

Overwriting /content/gst-rate-finder/gst_engine.py


In [ ]:
%%writefile /content/gst-rate-finder/barcode_engine.py
"""Barcode helpers. Barcode-to-item links live in data/barcodes.json and are added only
after you confirm the barcode belongs to that item. The GST rate always comes from the
verified item in data/gst_rates.json."""
from __future__ import annotations

import json
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
BARCODE_FILE = BASE_DIR / "data" / "barcodes.json"
PRODUCT_LENGTHS = (8, 12, 13)  # EAN-8, UPC-A, EAN-13


def is_product_code(code: str) -> bool:
    return (code or "").isdigit() and len(code) in PRODUCT_LENGTHS


def ean_valid(code: str) -> bool:
    if not is_product_code(code):
        return False
    digits = [int(c) for c in code]
    body, check = digits[:-1], digits[-1]
    total = 0
    for i, value in enumerate(body):
        position_from_right = len(body) - 1 - i
        weight = 3 if position_from_right % 2 == 0 else 1
        total += value * weight
    return (10 - total % 10) % 10 == check


def load_barcodes(path: Path = BARCODE_FILE) -> list[dict]:
    with open(path, encoding="utf-8") as fh:
        data = json.load(fh)
    records = data.get("records", [])
    for rec in records:
        code = rec.get("barcode", "")
        if not is_product_code(code):
            raise ValueError(f"Barcode '{code}' must be 8, 12 or 13 digits.")
        if not ean_valid(code):
            raise ValueError(f"Barcode '{code}' fails the check digit.")
        if not rec.get("record_id"):
            raise ValueError(f"Barcode '{code}' has no record_id.")
    return records


def lookup_barcode(code: str, barcodes: list[dict], records: list[dict]) -> dict | None:
    by_id = {r["id"]: r for r in records}
    for entry in barcodes:
        if entry["barcode"] == code:
            return by_id.get(entry["record_id"])
    return None


def decode_image(pil_image) -> list[str]:
    """Return the text of every barcode or QR code found in a PIL image."""
    try:
        from pyzbar.pyzbar import decode
    except (ImportError, OSError) as exc:
        raise RuntimeError(
            "Photo scanning needs pyzbar and libzbar0. Run the install cell again, "
            "or use Type code instead."
        ) from exc
    results = decode(pil_image.convert("RGB"))
    return [r.data.decode("utf-8", errors="replace") for r in results]

Writing /content/gst-rate-finder/barcode_engine.py


In [ ]:
%%writefile /content/gst-rate-finder/invoice_engine.py
"""Invoice tax calculation. Pure Python, so it can be tested."""
from __future__ import annotations

import re
from decimal import ROUND_HALF_UP, Decimal, InvalidOperation

HUNDRED = Decimal("100")
CENT = Decimal("0.01")
ZERO = Decimal("0")

# Format check only: state code, PAN-like block, entity code, Z, check character.
GSTIN_RE = re.compile(r"^\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]$")


def valid_gstin(gstin: str) -> bool:
    return bool(GSTIN_RE.match((gstin or "").strip().upper()))


def _d(value, label: str) -> Decimal:
    try:
        number = Decimal(str(value))
    except (InvalidOperation, ValueError):
        raise ValueError(f"{label} must be a number.") from None
    if not number.is_finite():
        raise ValueError(f"{label} must be a finite number.")
    return number


def _q(value: Decimal) -> Decimal:
    return value.quantize(CENT, rounding=ROUND_HALF_UP)


def _s(value: Decimal) -> str:
    return f"{value:.2f}"


def compute_invoice(lines: list[dict], supply: str) -> dict:
    """lines: description, hsn, qty, unit_price, gst_pct (total GST %).
    supply: 'intra' (CGST + SGST/UTGST) or 'inter' (IGST)."""
    if supply not in ("intra", "inter"):
        raise ValueError("Supply type must be intra-State or inter-State.")
    if not lines:
        raise ValueError("Add at least one item with a description.")

    out = []
    tot_taxable = tot_cgst = tot_sgst = tot_igst = tot_tax = tot_total = ZERO

    for i, ln in enumerate(lines, 1):
        desc = str(ln.get("description", "")).strip()
        if not desc:
            raise ValueError(f"Line {i}: description is required.")
        qty = _d(ln.get("qty"), f"Line {i}: quantity")
        price = _d(ln.get("unit_price"), f"Line {i}: unit price")
        rate = _d(ln.get("gst_pct"), f"Line {i}: GST %")
        if qty <= 0:
            raise ValueError(f"Line {i}: quantity must be greater than zero.")
        if price < 0:
            raise ValueError(f"Line {i}: unit price cannot be negative.")
        if rate < 0 or rate > HUNDRED:
            raise ValueError(f"Line {i}: GST % must be between 0 and 100.")

        taxable = _q(qty * price)
        tax = _q(taxable * rate / HUNDRED)
        if supply == "intra":
            cgst = _q(tax / 2)
            sgst = tax - cgst
            igst = ZERO
        else:
            cgst = ZERO
            sgst = ZERO
            igst = tax
        total = taxable + tax

        out.append({
            "description": desc,
            "hsn": str(ln.get("hsn", "")).strip(),
            "qty": format(qty.normalize(), "f"),
            "unit_price": _s(price),
            "taxable": _s(taxable),
            "gst_pct": format(rate.normalize(), "f"),
            "cgst": _s(cgst),
            "sgst": _s(sgst),
            "igst": _s(igst),
            "tax": _s(tax),
            "total": _s(total),
        })
        tot_taxable += taxable
        tot_cgst += cgst
        tot_sgst += sgst
        tot_igst += igst
        tot_tax += tax
        tot_total += total

    return {
        "lines": out,
        "taxable": _s(tot_taxable),
        "cgst": _s(tot_cgst),
        "sgst": _s(tot_sgst),
        "igst": _s(tot_igst),
        "total_tax": _s(tot_tax),
        "grand_total": _s(tot_total),
    }

Writing /content/gst-rate-finder/invoice_engine.py


In [ ]:
%%writefile /content/gst-rate-finder/invoice_pdf.py
"""Printable invoice PDF. Uses 'Rs.' because the standard PDF font has no rupee sign."""
from html import escape
from io import BytesIO

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


def build_invoice_pdf(seller: dict, buyer: dict, inv: dict, result: dict, disclaimer: str) -> bytes:
    buf = BytesIO()
    doc = SimpleDocTemplate(
        buf, pagesize=A4,
        leftMargin=14 * mm, rightMargin=14 * mm, topMargin=14 * mm, bottomMargin=14 * mm,
        title=f"Invoice {inv['number']}",
    )
    ss = getSampleStyleSheet()
    small = ParagraphStyle("small", parent=ss["Normal"], fontSize=8.5, leading=11)
    cell = ParagraphStyle("cell", parent=ss["Normal"], fontSize=8, leading=10)

    story = [
        Paragraph("TAX INVOICE (DRAFT)", ss["Title"]),
        Paragraph("Draft prepared with GST TAXORA. Check invoice content requirements under "
                  "GST law before issuing.", small),
        Spacer(1, 6),
    ]

    seller_block = (f"<b>Seller</b><br/><b>{escape(seller['name'])}</b><br/>"
                    f"{escape(seller['address'])}<br/>GSTIN: {escape(seller['gstin'] or 'Not provided')}")
    buyer_block = (f"<b>Buyer</b><br/><b>{escape(buyer['name'])}</b><br/>"
                   f"{escape(buyer['address'])}<br/>GSTIN: {escape(buyer['gstin'] or 'Not provided')}")
    parties = Table([[Paragraph(seller_block, small), Paragraph(buyer_block, small)]],
                    colWidths=[91 * mm, 91 * mm])
    parties.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.5, colors.grey),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
    ]))
    story += [parties, Spacer(1, 8)]

    supply_text = "Intra-State (CGST + SGST/UTGST)" if inv["supply"] == "intra" else "Inter-State (IGST)"
    story.append(Paragraph(
        f"<b>Invoice no.:</b> {escape(inv['number'])} &nbsp;&nbsp; <b>Date:</b> "
        f"{escape(inv['date'])} &nbsp;&nbsp; <b>Supply:</b> {supply_text}", small))
    story.append(Spacer(1, 8))

    header = ["Item", "HSN/SAC", "Qty", "Unit price", "Taxable", "GST %", "Tax", "Total"]
    rows = [header]
    for ln in result["lines"]:
        rows.append([
            Paragraph(escape(ln["description"]), cell),
            escape(ln["hsn"]),
            ln["qty"],
            f"Rs. {ln['unit_price']}",
            f"Rs. {ln['taxable']}",
            f"{ln['gst_pct']}%",
            f"Rs. {ln['tax']}",
            f"Rs. {ln['total']}",
        ])
    items = Table(rows, colWidths=[52 * mm, 20 * mm, 12 * mm, 22 * mm, 24 * mm, 14 * mm, 18 * mm, 20 * mm],
                  repeatRows=1)
    items.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#AF002D")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (2, 1), (-1, -1), "RIGHT"),
    ]))
    story += [items, Spacer(1, 10)]

    summary_rows = [["Taxable value", f"Rs. {result['taxable']}"]]
    if inv["supply"] == "intra":
        summary_rows += [["CGST", f"Rs. {result['cgst']}"], ["SGST/UTGST", f"Rs. {result['sgst']}"]]
    else:
        summary_rows.append(["IGST", f"Rs. {result['igst']}"])
    summary_rows += [["Total tax", f"Rs. {result['total_tax']}"],
                     ["Grand total", f"Rs. {result['grand_total']}"]]
    summary = Table(summary_rows, colWidths=[50 * mm, 40 * mm], hAlign="RIGHT")
    summary.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("ALIGN", (1, 0), (1, -1), "RIGHT"),
        ("FONTNAME", (0, -1), (-1, -1), "Helvetica-Bold"),
    ]))
    story += [summary, Spacer(1, 12)]

    story.append(Paragraph("Amounts are rounded to 2 decimal places (half-up). In intra-State supply, "
                           "CGST is half the tax and SGST takes any paise remainder.", small))
    story.append(Spacer(1, 6))
    story.append(Paragraph(escape(disclaimer), small))

    doc.build(story)
    return buf.getvalue()

Writing /content/gst-rate-finder/invoice_pdf.py


In [ ]:
%%writefile /content/gst-rate-finder/guides.py
"""Step-by-step guides and video slots.
To add a video, paste a YouTube or other link into the "url" field in this file.
Leave it as "" until you have a real link that you have checked."""

VIDEO_NOTE = "Video coming soon. The link will appear here once it is added in guides.py."

GUIDES = {
    "Getting started": {
        "title": "How to use GST TAXORA",
        "summary": "A quick tour for first-time users. You do not need an account or personal details.",
        "steps": [
            "Click **Get started** on the first screen.",
            "On the menu, choose the feature you need. Each option is a button with a short description.",
            "The selected feature opens on its own page. Use **Back to menu** at the top to return.",
            "Read the disclaimer in the sidebar (open it with the » arrow at the top left). GST advice depends on the facts of your case.",
            "Always check the official notification before you invoice or file a return.",
        ],
        "videos": [{"title": "Overview of GST TAXORA", "url": ""}],
    },
    "Search a GST rate": {
        "title": "Search for a goods or service rate",
        "summary": "Find the GST rate, HSN or SAC code and notification for an item.",
        "steps": [
            "Open **Search** from the menu.",
            "Pick what you are looking for: goods by name, goods by HSN code, or services by name or SAC code.",
            "Choose **Intra-State** if buyer and seller are in the same state, or **Inter-State** if they are not.",
            "Type the item name, for example *rice* or *cement*. Small spelling mistakes are fine.",
            "Read the result card: status, total GST, the CGST/SGST or IGST split, the notification and the condition.",
            "Suggestions appear in a separate box and are not confirmed matches. Check the item name before relying on them.",
            "Use **Download these results (CSV)** to save the result.",
        ],
        "videos": [{"title": "How to search a rate", "url": ""}],
    },
    "Browse by category": {
        "title": "Browse items by category and GST slab",
        "summary": "See all items in a category or at a particular rate.",
        "steps": [
            "Open **Browse by category** from the menu.",
            "Choose a category, or leave it as *All categories*.",
            "Choose Goods, Services or Both.",
            "Choose a GST slab to see only items at that rate.",
            "Use Search to see the full condition and notes for any item in the table.",
        ],
        "videos": [{"title": "How to browse by category", "url": ""}],
    },
    "GST calculator": {
        "title": "Calculate GST on an amount",
        "summary": "Add GST to a price, or take GST out of a price that already includes it.",
        "steps": [
            "Open **GST calculator** from the menu.",
            "Enter the base amount in rupees.",
            "Choose a rate: pick an item from the dataset, or choose *Manual rate* and type the rate yourself.",
            "Choose Intra-State or Inter-State.",
            "Choose **Add GST to base amount** if your price excludes GST, or **Extract GST** if it includes GST.",
            "Read the taxable value, CGST, SGST/UTGST, IGST and final amount.",
        ],
        "videos": [{"title": "How to use the calculator", "url": ""}],
    },
    "Barcode & QR scanner": {
        "title": "Scan a product barcode or QR code",
        "summary": "Scan a product to see its category and GST rate, if that product is linked in the app.",
        "steps": [
            "Open **Barcode & QR scanner** from the menu.",
            "Choose Intra-State or Inter-State.",
            "Use the **Camera** tab and allow camera access when your browser asks. Hold the barcode steady and well lit.",
            "Or use **Upload photo** to choose a clear photo of the barcode.",
            "Or use **Type code** and enter the number printed under the barcode.",
            "If the product is linked to a verified item, its category and GST rate appear. If not, search by name instead.",
        ],
        "videos": [{"title": "How to scan a barcode", "url": ""}],
    },
    "GST invoice generator": {
        "title": "Create a printable invoice",
        "summary": "Enter seller and buyer details, add items, and download a PDF draft.",
        "steps": [
            "Open **GST invoice generator** from the menu.",
            "Enter the seller and buyer names and addresses. GSTIN is optional for unregistered parties.",
            "Enter an invoice number and date.",
            "Choose Intra-State or Inter-State.",
            "In the item table, add a row for each item: description, HSN/SAC, quantity, unit price and GST %.",
            "Use **Look up GST %** above the table to copy the correct rate from the dataset.",
            "Check the summary: taxable value, CGST, SGST/UTGST or IGST, and grand total.",
            "Click **Download invoice PDF**. The PDF is marked DRAFT. Check invoice content rules under GST law before issuing it.",
        ],
        "videos": [{"title": "How to make an invoice", "url": ""}],
    },
    "Bulk check": {
        "title": "Check many items at once",
        "summary": "Upload a list of items in a CSV file and get the GST rate for each one.",
        "steps": [
            "Open **Bulk check** from the menu.",
            "Download the sample CSV, or make your own file with one column named **item**.",
            "Upload the CSV file.",
            "Read the table. Each row shows the first matched goods item, its HSN code, total GST and status.",
            "Rows marked *Not found* need a better name. Try Search for those items.",
            "Download the results as CSV.",
        ],
        "videos": [{"title": "How to use bulk check", "url": ""}],
    },
    "Privacy & data": {
        "title": "Your privacy and data",
        "summary": "What the app keeps, what it does not keep, and how to delete your data.",
        "steps": [
            "Open **Privacy & data** from the menu to read the full notice.",
            "Searches and calculations need no personal data.",
            "Recent searches are off by default. If you turn them on, they stay only in your session.",
            "Invoice details stay in your session. Click **Delete all session data now** to clear them.",
        ],
        "videos": [],
    },
}

Writing /content/gst-rate-finder/guides.py


In [ ]:
%%writefile /content/gst-rate-finder/data/gst_rates.json
{
  "meta": {
    "dataset_version": "0.1.1-initial",
    "last_updated": "2026-11-18",
    "status": "Compiled from the uploaded CBIC notification texts (9/2025, 10/2025, 15/2025, 11/2017 with 2026 amendments noted). Not yet checked line-by-line against the live CBIC portal. Confirm each entry before relying on it.",
    "source": "https://cbic-gst.gov.in/",
    "change_log": [
      {"date": "2026-11-18", "who": "initial build", "what": "30 sample records added from uploaded notification texts", "source": "9/2025, 10/2025, 15/2025, 11/2017-CT(Rate) and 2026 amendments in project files"}
    ]
  },
  "records": [
    {"id":"G001","kind":"goods","name":"Rice (pre-packaged and labelled)","aliases":["rice","basmati rice","chawal","packaged rice"],"hsn_sac":"1006","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 48","condition":"Pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G002","kind":"goods","name":"Rice (loose, not pre-packaged)","aliases":["rice loose","loose rice","open rice"],"hsn_sac":"1006","category":"Food & Agriculture","treatment":"exempt","cgst_pct":"0","notification":"10/2025-Central Tax (Rate)","serial":"S.No 67","condition":"Other than pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Exempt, so no tax is charged."},
    {"id":"G003","kind":"goods","name":"Wheat (pre-packaged and labelled)","aliases":["wheat","gehu","wheat grain"],"hsn_sac":"1001","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 43","condition":"Pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G004","kind":"goods","name":"Wheat or meslin flour (atta, maida), pre-packaged","aliases":["atta","wheat flour","maida","flour"],"hsn_sac":"1101","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 51","condition":"Pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G005","kind":"goods","name":"Fresh, pasteurised or UHT milk","aliases":["milk","fresh milk","pasteurised milk","uht milk","doodh"],"hsn_sac":"0401","category":"Food & Agriculture","treatment":"exempt","cgst_pct":"0","notification":"10/2025-Central Tax (Rate)","serial":"S.No 15","condition":"Not concentrated and no added sugar","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G006","kind":"goods","name":"Milk and cream, concentrated or sweetened (incl. condensed milk)","aliases":["condensed milk","sweetened milk","concentrated milk"],"hsn_sac":"0402","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 4","condition":"Concentrated or containing added sugar","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G007","kind":"goods","name":"Butter, ghee and dairy fats","aliases":["butter","ghee","dairy spread"],"hsn_sac":"0405","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 7","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G008","kind":"goods","name":"Chena or paneer","aliases":["paneer","chena"],"hsn_sac":"0406","category":"Food & Agriculture","treatment":"exempt","cgst_pct":"0","notification":"10/2025-Central Tax (Rate)","serial":"S.No 17","condition":"Whether or not pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G009","kind":"goods","name":"Cheese (other than chena or paneer)","aliases":["cheese"],"hsn_sac":"0406","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 8","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G010","kind":"goods","name":"Natural honey (pre-packaged and labelled)","aliases":["honey","shahad"],"hsn_sac":"0409","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 10","condition":"Pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Loose honey is exempt under 10/2025, S.No 19."},
    {"id":"G011","kind":"goods","name":"Tea (processed)","aliases":["tea","chai","chai patti","tea leaves"],"hsn_sac":"0902","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 34","condition":"Other than unprocessed green leaves","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G012","kind":"goods","name":"Coffee, roasted","aliases":["coffee","roasted coffee"],"hsn_sac":"0901","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 33","condition":"Roasted, not raw beans","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Coffee beans not roasted are exempt under 10/2025, S.No 57."},
    {"id":"G013","kind":"goods","name":"Salt (including table salt)","aliases":["salt","table salt","namak"],"hsn_sac":"2501","category":"Food & Agriculture","treatment":"exempt","cgst_pct":"0","notification":"10/2025-Central Tax (Rate)","serial":"S.No 109","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G014","kind":"goods","name":"Sugar (cane or beet, pre-packaged)","aliases":["sugar","cheeni"],"hsn_sac":"1701","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 105","condition":"Other than jaggery and khandsari","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G015","kind":"goods","name":"Jaggery (gur), pre-packaged and labelled","aliases":["jaggery","gur","gud"],"hsn_sac":"1701/1702","category":"Food & Agriculture","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 106","condition":"Pre-packaged and labelled","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Jaggery other than pre-packaged is exempt under 10/2025, S.No 96."},
    {"id":"G016","kind":"goods","name":"Portland cement","aliases":["cement"],"hsn_sac":"2523","category":"Construction & Building","treatment":"taxable","cgst_pct":"9","notification":"9/2025-Central Tax (Rate)","serial":"Schedule II, S.No 9","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G017","kind":"goods","name":"Articles of jewellery of precious metal (gold, silver)","aliases":["gold jewellery","gold","jewellery","jewelry","gold ornament","silver jewellery"],"hsn_sac":"7113","category":"Precious Metals & Jewellery","treatment":"taxable","cgst_pct":"1.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule IV, S.No 10","condition":"Articles of jewellery of precious metal","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Unwrought gold and silver are in Schedule IV, S.Nos 5 and 3. Check the notification before invoicing bullion."},
    {"id":"G018","kind":"goods","name":"Rough or simply sawn diamonds","aliases":["diamond","diamonds","rough diamond"],"hsn_sac":"7102","category":"Precious Metals & Jewellery","treatment":"taxable","cgst_pct":"0.125","notification":"9/2025-Central Tax (Rate)","serial":"Schedule V, S.No 1","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G019","kind":"goods","name":"Automatic data processing machines (laptops, computers)","aliases":["laptop","computer","desktop","notebook","pc","laptop computer"],"hsn_sac":"8471","category":"Electronics","treatment":"taxable","cgst_pct":"9","notification":"9/2025-Central Tax (Rate)","serial":"Schedule II, S.No 456","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Confirm the Schedule for your exact model before invoicing."},
    {"id":"G020","kind":"goods","name":"Drugs and medicines (not in Annexure I of 10/2025)","aliases":["medicine","medicines","drug","drugs","tablet","tablets","medicament"],"hsn_sac":"30","category":"Medicines & Healthcare","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 226","condition":"Other than items at nil under 10/2025, Annexure I (S.No 113)","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Check the 10/2025 Annexure I list for specific drugs at nil rate."},
    {"id":"G021","kind":"goods","name":"Electrically operated vehicles (incl. e-2W, e-3W, e-bicycles)","aliases":["electric vehicle","ev","electric scooter","e scooter","electric bike","e bike","ebike","electric car"],"hsn_sac":"87","category":"Automobiles","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 441","condition":"Run solely on electrical energy (per the notification's explanation)","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Listed at chapter level (HSN 87). Confirm the exact vehicle category on the notification."},
    {"id":"G022","kind":"goods","name":"Bicycles and other non-motorised cycles","aliases":["bicycle","cycle","cycles"],"hsn_sac":"8712","category":"Automobiles","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 456","condition":"Non-motorised","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G023","kind":"goods","name":"Motor cars (other than small cars in Schedule II)","aliases":["car","motor car","cars","petrol car","diesel car","suv","sedan","hatchback"],"hsn_sac":"8703","category":"Automobiles","treatment":"taxable","cgst_pct":"20","notification":"9/2025-Central Tax (Rate)","serial":"Schedule III, S.No 5","condition":"Excludes items at Schedule II, S.Nos 533 to 539 (ambulances and listed small petrol, CNG, LPG and diesel cars)","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Small cars and some EVs fall under other entries. Verify the exact vehicle before invoicing."},
    {"id":"G024","kind":"goods","name":"Motorcycles (engine capacity above 350 cc)","aliases":["motorcycle","bike","motor bike"],"hsn_sac":"8711","category":"Automobiles","treatment":"taxable","cgst_pct":"20","notification":"9/2025-Central Tax (Rate)","serial":"Schedule III, S.No 8","condition":"Engine capacity exceeding 350 cc","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Motorcycles up to 350 cc are not in this dataset yet."},
    {"id":"G025","kind":"goods","name":"Cigarettes, cigars, biris and other tobacco products","aliases":["cigarette","cigarettes","cigar","tobacco","biri","biris"],"hsn_sac":"2402/2403","category":"Tobacco & Pan Masala","treatment":"taxable","cgst_pct":"14","notification":"9/2025-Central Tax (Rate)","serial":"Schedule VII, S.Nos 3 and 4","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Compensation cess on tobacco is set by a separate notification that is not in the uploaded files. Verify before use.","cess":"Compensation cess on tobacco is set by a separate notification not in the uploaded files - verify"},
    {"id":"G026","kind":"goods","name":"Footwear (sale value up to Rs 2,500 per pair)","aliases":["footwear","shoes","shoe","chappal","sandal","sandals","slipper"],"hsn_sac":"64","category":"Textiles & Apparel","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 392","condition":"Sale value not exceeding Rs 2,500 per pair","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G027","kind":"goods","name":"Footwear (sale value above Rs 2,500 per pair)","aliases":["footwear","shoes","shoe","chappal","sandal","sandals","slipper"],"hsn_sac":"6401/6402/6403/6404/6405/6406","category":"Textiles & Apparel","treatment":"taxable","cgst_pct":"9","notification":"9/2025-Central Tax (Rate)","serial":"Schedule II, S.Nos 202 to 207","condition":"Sale value above Rs 2,500 per pair","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Confirm the price threshold entry by entry on the notification."},
    {"id":"G028","kind":"goods","name":"Apparel and clothing accessories (sale value up to Rs 2,500 per piece)","aliases":["clothes","clothing","garment","garments","apparel","shirt","t shirt","tshirt","kurta"],"hsn_sac":"61/62","category":"Textiles & Apparel","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.Nos 388 and 389","condition":"Sale value up to Rs 2,500 per piece","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G029","kind":"goods","name":"Apparel and clothing accessories (sale value above Rs 2,500 per piece)","aliases":["clothes","clothing","garment","garments","apparel","shirt","t shirt","tshirt","kurta"],"hsn_sac":"61/62","category":"Textiles & Apparel","treatment":"taxable","cgst_pct":"9","notification":"9/2025-Central Tax (Rate)","serial":"Schedule II, S.Nos 197 and 198","condition":"Sale value above Rs 2,500 per piece","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"G030","kind":"goods","name":"Solar water heater and system","aliases":["solar water heater","solar heater","water heater solar"],"hsn_sac":"8419 12","category":"Electronics","treatment":"taxable","cgst_pct":"2.5","notification":"9/2025-Central Tax (Rate)","serial":"Schedule I, S.No 428","condition":"","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"S001","kind":"service","name":"Restaurant service (no air-conditioning, no liquor licence)","aliases":["restaurant","eating joint","dhaba","food service"],"hsn_sac":"9963","category":"Hotels & Food Services","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate), as amended by 15/2025-Central Tax (Rate)","serial":"Entry 7(i), heading 9963","condition":"No air-conditioning or central air-heating at any time in the year, and no liquor licence","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Rate revised by 15/2025 with effect from 22 Sep 2025."},
    {"id":"S002","kind":"service","name":"Restaurant service (air-conditioned or licensed to serve liquor)","aliases":["restaurant","ac restaurant","bar restaurant","restaurant with bar","hotel restaurant"],"hsn_sac":"9963","category":"Hotels & Food Services","treatment":"taxable","cgst_pct":"9","notification":"11/2017-Central Tax (Rate)","serial":"Entries 7(iii) and 7(iv), heading 9963","condition":"Air-conditioned or centrally air-heated, or licensed to serve alcoholic liquor","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"S003","kind":"service","name":"Air travel, economy class","aliases":["flight","air ticket","airfare","air travel","air fare","plane ticket"],"hsn_sac":"9964","category":"Transport","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate)","serial":"Entry 8(iii), heading 9964","condition":"Economy class; ITC proviso applies (goods used in supply not credited)","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":"Air travel other than economy class is a separate entry, not in this dataset yet."},
    {"id":"S004","kind":"service","name":"Rail travel, first class or air-conditioned coach","aliases":["train","rail","railway ticket","train ticket","ac coach","first class train"],"hsn_sac":"9964","category":"Transport","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate)","serial":"Entry 8(i), heading 9964","condition":"First class or air-conditioned coach; ITC proviso applies","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":"Other rail classes are not in this dataset yet."},
    {"id":"S005","kind":"service","name":"Goods transport agency (road freight)","aliases":["gta","goods transport agency","transport","freight","truck freight","lorry","logistics transport"],"hsn_sac":"9965","category":"Transport","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate)","serial":"Entry 9(iii), heading 9965","condition":"Issues a consignment note; ITC proviso applies","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":"The GTA definition was amended by 06/2026 with effect from 1 Nov 2026. Verify the current definition."},
    {"id":"S006","kind":"service","name":"Legal services (advocates)","aliases":["lawyer","advocate","legal","legal services","law firm"],"hsn_sac":"9982","category":"Professional Services","treatment":"taxable","cgst_pct":"9","notification":"11/2017-Central Tax (Rate)","serial":"Entry 20, heading 9982","condition":"","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"S007","kind":"service","name":"Accounting and auditing services","aliases":["ca","chartered accountant","accounting","audit","auditing","bookkeeping"],"hsn_sac":"9982","category":"Professional Services","treatment":"taxable","cgst_pct":"9","notification":"11/2017-Central Tax (Rate)","serial":"Entry 20, heading 9982","condition":"","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"S008","kind":"service","name":"Tour operator services","aliases":["tour","tour operator","travel package","tour package","holiday package"],"hsn_sac":"9985","category":"Travel & Tourism","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate)","serial":"Entry 23(i), heading 9985","condition":"Bill shows the inclusive gross amount for accommodation and transport; ITC proviso applies","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":""},
    {"id":"S009","kind":"service","name":"Beauty and physical well-being services (salon, spa)","aliases":["salon","spa","beauty parlour","beauty parlor","parlour","haircut","beauty services","massage"],"hsn_sac":"9997","category":"Personal Services","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate), as amended by 15/2025-Central Tax (Rate)","serial":"Entry 35(i), heading 9997","condition":"Group 99972 services; ITC proviso applies","effective_from":"2025-09-22","source_url":"https://cbic-gst.gov.in/","note":"Other washing, cleaning and dyeing services are in entry 35(ii) at 9%."},
    {"id":"S010","kind":"service","name":"Transport of passengers by motorcab (fuel included)","aliases":["cab","taxi","motorcab","cab fare","taxi fare"],"hsn_sac":"9964","category":"Transport","treatment":"taxable","cgst_pct":"2.5","notification":"11/2017-Central Tax (Rate)","serial":"Entry 8(vi), heading 9964","condition":"Cost of fuel is included in the consideration charged to the recipient","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":"Renting of motorcab with fuel is entry 10(i) at 2.5%."},
    {"id":"S011","kind":"service","name":"Construction of a residential building or complex for sale","aliases":["construction","building","flat","apartment","house construction","real estate construction"],"hsn_sac":"9954","category":"Construction & Building","treatment":"taxable","cgst_pct":"9","notification":"11/2017-Central Tax (Rate)","serial":"Entry 3(i), heading 9954","condition":"Applies until the completion certificate or first occupation, whichever is earlier, for the entire consideration received","effective_from":"2017-07-01","source_url":"https://cbic-gst.gov.in/","note":"Valuation rules in paragraph 2 of the notification apply. Confirm with a tax professional before invoicing."}
  ]
}

Writing /content/gst-rate-finder/data/gst_rates.json


In [ ]:
%%writefile /content/gst-rate-finder/data/barcodes.json
{
  "meta": {
    "version": "0.1.0",
    "note": "Barcode-to-item links. Add an entry only after you have confirmed that the barcode belongs to the item. record_id must match an id in gst_rates.json. Empty on purpose."
  },
  "records": []
}

Writing /content/gst-rate-finder/data/barcodes.json


In [ ]:
%%writefile /content/gst-rate-finder/app.py
import datetime as dt
import re

import pandas as pd
import streamlit as st

from barcode_engine import decode_image, ean_valid, is_product_code, load_barcodes, lookup_barcode
from gst_engine import (
    DISCLAIMER,
    NOT_FOUND_MSG,
    calculate,
    digits_only,
    fmt_pct,
    load_dataset,
    rate_split,
    result_row,
    search_code,
    search_text,
    status_text,
    to_csv,
    total_gst_pct,
)
from guides import GUIDES, VIDEO_NOTE
from invoice_engine import compute_invoice, valid_gstin
from invoice_pdf import build_invoice_pdf

APP_NAME = "GST TAXORA"
MAX_HISTORY = 10
TODAY = dt.date.today().isoformat()
SUPPLY_OPTIONS = ["Intra-State (CGST + SGST/UTGST)", "Inter-State (IGST)"]

st.set_page_config(page_title=APP_NAME, page_icon="🧾", layout="wide",
                   initial_sidebar_state="collapsed")

CSS = """
<style>
.block-container { padding-top: 2rem; max-width: 1100px; }
.splash { text-align: center; padding: 8rem 1rem 3rem 1rem; }
.splash h1 {
    font-size: 5rem; font-weight: 900; letter-spacing: 3px; margin: 0;
    background: linear-gradient(90deg, #FF7A18, #AF002D, #319197);
    -webkit-background-clip: text; background-clip: text; color: transparent;
}
div.stButton > button {
    min-height: 3.4rem; border-radius: 14px; font-size: 1.05rem; font-weight: 600;
}
[data-testid="stMetric"] {
    background: rgba(175, 0, 45, 0.08); padding: 0.7rem; border-radius: 12px;
}
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)


@st.cache_data
def get_dataset():
    return load_dataset()


@st.cache_data
def get_barcodes():
    return load_barcodes()


META, RECORDS = get_dataset()
BARCODES = get_barcodes()


def blank_rows():
    return [{"Description": "", "HSN/SAC": "", "Qty": 1.0, "Unit price (Rs)": 0.0, "GST %": 18.0}]


for key, value in {"page": "splash", "history": [], "inv_rows": blank_rows()}.items():
    st.session_state.setdefault(key, value)

MENU = [
    ("search", "🔎", "Search", "Find the GST rate, HSN or SAC code by name or code"),
    ("browse", "🗂️", "Browse by category", "Filter items by category and GST slab"),
    ("calc", "🧮", "GST calculator", "Add or extract GST, with the CGST/SGST/IGST split"),
    ("scan", "📷", "Barcode & QR scanner", "Scan a product code to get its category and GST rate"),
    ("invoice", "🧾", "GST invoice generator", "Create a printable invoice PDF with GST"),
    ("bulk", "📑", "Bulk check", "Check many items from a CSV file at once"),
    ("guide", "📚", "Step-by-step guide", "Learn each feature with steps and videos"),
    ("privacy", "🔒", "Privacy & data", "See what is kept and delete your session data"),
]
MENU_TITLE = {key: title for key, _, title, _ in MENU}
MENU_ICON = {key: icon for key, icon, _, _ in MENU}


def go(page):
    st.session_state.page = page


def page_header(key):
    left, right = st.columns([1.3, 5])
    left.button("← Back to menu", key=f"back_{key}", on_click=go, args=("menu",))
    right.markdown(f"## {MENU_ICON[key]} {MENU_TITLE[key]}")


def supply_key(label: str) -> str:
    return "intra" if label.startswith("Intra") else "inter"


def remember(entry: str) -> None:
    history = st.session_state.history
    if entry not in history:
        history.insert(0, entry)
        del history[MAX_HISTORY:]


def show_result(rec: dict, supply: str) -> None:
    pct = total_gst_pct(rec)
    split = rate_split(rec)
    with st.container(border=True):
        kind_label = "Service (SAC)" if rec["kind"] == "service" else "Goods (HSN)"
        st.subheader(rec["name"])
        st.caption(f"{kind_label}: {rec['hsn_sac']} · Category: {rec['category']}")
        if rec["treatment"] == "taxable":
            st.success("GST APPLICABLE")
        else:
            st.info("GST NOT APPLICABLE — Exempt (nil rate under the exemption notification)")

        if supply == "intra":
            c1, c2, c3, c4 = st.columns(4)
            c1.metric("Total GST", fmt_pct(pct))
            c2.metric("CGST", fmt_pct(split["cgst"]))
            c3.metric("SGST/UTGST", fmt_pct(split["sgst"]))
            c4.metric("IGST", "Not applicable")
        else:
            c1, c2 = st.columns(2)
            c1.metric("Total GST", fmt_pct(pct))
            c2.metric("IGST", fmt_pct(split["igst"]))

        st.markdown(f"**Cess:** {rec.get('cess', 'Not covered by this dataset - verify on CBIC')}")
        st.markdown(f"**Notification:** {rec['notification']} · **Entry:** {rec['serial']}")
        st.markdown(f"**Condition:** {rec['condition']}")
        st.markdown(f"**Effective from:** {rec['effective_from']}")
        st.markdown(f"**Source:** [CBIC GST portal]({rec['source_url']})")
        if rec["note"]:
            st.caption(f"Note: {rec['note']}")


def history_panel() -> None:
    with st.expander("Recent searches (this session)"):
        if not st.session_state.history:
            st.write("No recent searches.")
        else:
            for entry in st.session_state.history:
                st.write(f"- {entry}")
        st.caption("Kept only in this session's server memory. Not saved to disk or browser storage.")
        if st.button("Clear history", key="clear_history"):
            st.session_state.history = []
            st.rerun()


def splash():
    st.markdown('<div class="splash"><h1>GST TAXORA</h1></div>', unsafe_allow_html=True)
    _, mid, _ = st.columns([1, 1, 1])
    mid.button("Get started  →", use_container_width=True, on_click=go, args=("menu",), key="start")


def menu_page():
    st.markdown("<h2 style='text-align:center'>What would you like to do?</h2>", unsafe_allow_html=True)
    st.write("")
    cols = st.columns(2)
    for i, (key, icon, title, desc) in enumerate(MENU):
        with cols[i % 2]:
            st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                      on_click=go, args=(key,))
            st.caption(desc)


def search_page():
    page_header("search")
    mode = st.radio("What are you looking up?",
                    ["Goods by name", "Goods by HSN code", "Services by name or SAC code"],
                    horizontal=True, key="search_mode")
    supply = supply_key(st.radio("Supply type", SUPPLY_OPTIONS, horizontal=True, key="supply_type"))
    keep = st.checkbox("Keep my recent searches for this session (off by default)",
                       value=False, key="keep_history")
    query = st.text_input("Item, HSN code or SAC code", key="query",
                          placeholder="e.g. rice, ricee, 1006, cement, restaurant, 9963")
    history_panel()

    if not query.strip():
        st.caption("Type a name (small spelling mistakes are fine) or a 4, 6 or 8-digit code.")
        return

    try:
        if mode == "Goods by HSN code":
            exact, suggestions = search_code(query, RECORDS, "goods"), []
        elif mode == "Goods by name":
            exact, suggestions = search_text(query, RECORDS, "goods")
        elif digits_only(query):
            exact, suggestions = search_code(query, RECORDS, "service"), []
        else:
            exact, suggestions = search_text(query, RECORDS, "service")
    except ValueError as err:
        st.error(str(err))
        return

    if not exact and not suggestions:
        st.warning(NOT_FOUND_MSG)
        return
    if keep:
        remember(f"{query.strip()} [{mode}]")

    for rec in exact[:10]:
        show_result(rec, supply)
    if len(exact) > 10:
        st.caption(f"Showing the first 10 of {len(exact)} matches. Refine your search to narrow it.")
    if suggestions:
        st.info("Suggestions only — these are NOT confirmed classifications.")
        for rec in suggestions[:5]:
            st.write(f"- {rec['name']} (HSN/SAC {rec['hsn_sac']})")
    if exact:
        rows = [result_row(r, supply, META, TODAY) for r in exact]
        st.download_button("Download these results (CSV)", data=to_csv(rows),
                           file_name=f"gst_results_{TODAY}.csv", mime="text/csv", key="dl_results")


def browse_page():
    page_header("browse")
    categories = ["All categories"] + sorted({r["category"] for r in RECORDS})
    slabs = {"All rates": None}
    for pct in sorted({total_gst_pct(r) for r in RECORDS}):
        slabs["0% (exempt / nil)" if pct == 0 else fmt_pct(pct)] = pct

    c1, c2, c3 = st.columns(3)
    category = c1.selectbox("Category", categories, key="browse_cat")
    kind = c2.radio("Type", ["Both", "Goods", "Services"], horizontal=True, key="browse_kind")
    slab_label = c3.selectbox("GST slab (from the dataset)", list(slabs), key="browse_slab")

    rows = []
    for rec in RECORDS:
        if category != "All categories" and rec["category"] != category:
            continue
        if kind == "Goods" and rec["kind"] != "goods":
            continue
        if kind == "Services" and rec["kind"] != "service":
            continue
        if slabs[slab_label] is not None and total_gst_pct(rec) != slabs[slab_label]:
            continue
        rows.append({
            "Item": rec["name"],
            "Type": "Service" if rec["kind"] == "service" else "Goods",
            "HSN/SAC": rec["hsn_sac"],
            "Status": "Applicable" if rec["treatment"] == "taxable" else "Exempt",
            "Total GST": fmt_pct(total_gst_pct(rec)),
            "Notification": rec["notification"],
            "Entry": rec["serial"],
        })
    if not rows:
        st.info("No items match these filters.")
        return
    st.dataframe(pd.DataFrame(rows), hide_index=True, use_container_width=True)
    st.caption("Use Search to see the full condition and notes for an item.")


def calc_page():
    page_header("calc")
    st.caption("An estimate based on the classification you choose. Rounded to 2 decimal places.")
    taxable = [r for r in RECORDS if r["treatment"] == "taxable"]
    labels = ["Manual rate"] + [f"{r['name']} — {fmt_pct(total_gst_pct(r))}" for r in taxable]

    c1, c2 = st.columns(2)
    base = c1.number_input("Base amount (₹)", min_value=0.01, value=1000.0, step=100.0,
                           format="%.2f", key="calc_base")
    choice = c2.selectbox("Rate source", labels, key="calc_choice")

    if choice == "Manual rate":
        rate = st.number_input("GST rate (%)", min_value=0.0, max_value=100.0, value=18.0,
                               step=0.25, format="%.3f", key="calc_rate")
        st.caption("Manual rate: you are responsible for confirming that it applies to your item.")
    else:
        rec = taxable[labels.index(choice) - 1]
        rate = float(total_gst_pct(rec))
        st.caption(f"Rate taken from {rec['notification']}, {rec['serial']}.")

    c3, c4 = st.columns(2)
    supply = supply_key(c3.radio("Supply type", SUPPLY_OPTIONS, key="calc_supply"))
    mode_label = c4.radio("Mode", ["Add GST to base amount", "Extract GST from inclusive amount"],
                          key="calc_mode")
    mode = "add" if mode_label.startswith("Add") else "extract"

    try:
        res = calculate(base, rate, supply, mode)
    except ValueError as err:
        st.error(str(err))
        return

    m1, m2, m3 = st.columns(3)
    m1.metric("Taxable value", f"₹{res['taxable_value']}")
    m2.metric("CGST", f"₹{res['cgst']}")
    m3.metric("SGST/UTGST", f"₹{res['sgst_utgst']}")
    m4, m5, m6 = st.columns(3)
    m4.metric("IGST", f"₹{res['igst']}")
    m5.metric("Total tax", f"₹{res['total_tax']}")
    m6.metric("Final amount", f"₹{res['final_amount']}")
    st.caption(res["rounding_note"])

    export = [{
        **res,
        "rate_pct": str(rate),
        "supply": "Intra-State" if supply == "intra" else "Inter-State",
        "mode": "Add GST" if mode == "add" else "Extract GST",
        "dataset_version": META["dataset_version"],
        "downloaded_on": TODAY,
        "disclaimer": DISCLAIMER,
    }]
    st.download_button("Download calculation (CSV)", data=to_csv(export),
                       file_name=f"gst_calc_{TODAY}.csv", mime="text/csv", key="dl_calc")


def handle_code(code: str, supply: str):
    st.markdown(f"**Code read:** `{code}`")
    if not is_product_code(code):
        st.info("This is not an EAN-8, UPC-A or EAN-13 product barcode, so no product GST rate can be shown.")
        return
    if not ean_valid(code):
        st.warning("The check digit does not match. Scan again or type the number carefully.")
        return
    rec = lookup_barcode(code, BARCODES, RECORDS)
    if rec is None:
        st.info("This barcode is valid but is not yet linked to a verified item, so no rate is shown. "
                "Use Search to find the item by name.")
        return
    st.success(f"Category: {rec['category']}")
    show_result(rec, supply)


def scan_page():
    page_header("scan")
    supply = supply_key(st.radio("Supply type", SUPPLY_OPTIONS, horizontal=True, key="scan_supply"))
    tab_cam, tab_photo, tab_type = st.tabs(["📸 Camera", "🖼️ Upload photo", "⌨️ Type code"])

    with tab_cam:
        cam = st.camera_input("Point the camera at the barcode or QR code", key="scan_cam")
    with tab_photo:
        up = st.file_uploader("Upload a clear photo of the barcode or QR code",
                              type=["png", "jpg", "jpeg"], key="scan_up")
    with tab_type:
        typed = st.text_input("Barcode number (EAN-8, UPC-A or EAN-13)", key="scan_typed")

    codes: list[str] = []
    source = cam if cam is not None else up
    if source is not None:
        try:
            from PIL import Image
            codes = decode_image(Image.open(source))
            if not codes:
                st.warning("No barcode or QR code found in this image. Try a closer, sharper photo.")
        except RuntimeError as err:
            st.error(str(err))
        except Exception:
            st.warning("Could not read this image. Try another photo or type the number.")

    if typed.strip():
        codes = [typed.strip()]
    for code in codes:
        handle_code(code, supply)


def _txt(value) -> str:
    return "" if value is None or pd.isna(value) else str(value).strip()


def _num(value, default: float) -> float:
    return default if value is None or pd.isna(value) else float(value)


def clear_invoice():
    for key in ["inv_s_name", "inv_s_gstin", "inv_s_addr", "inv_b_name", "inv_b_gstin",
                "inv_b_addr", "inv_no", "inv_date", "inv_supply", "inv_editor"]:
        st.session_state.pop(key, None)
    st.session_state.inv_rows = blank_rows()


def invoice_page():
    page_header("invoice")
    st.info("Details you enter stay in this session and are used only to build the PDF.")

    c1, c2 = st.columns(2)
    with c1:
        st.markdown("**Seller**")
        s_name = st.text_input("Seller name", key="inv_s_name")
        s_gstin = st.text_input("Seller GSTIN (optional)", key="inv_s_gstin")
        s_addr = st.text_area("Seller address", key="inv_s_addr", height=80)
    with c2:
        st.markdown("**Buyer**")
        b_name = st.text_input("Buyer name", key="inv_b_name")
        b_gstin = st.text_input("Buyer GSTIN (optional)", key="inv_b_gstin")
        b_addr = st.text_area("Buyer address", key="inv_b_addr", height=80)

    if s_gstin.strip() and not valid_gstin(s_gstin):
        st.warning("Seller GSTIN format looks wrong. A GSTIN has 15 characters.")
    if b_gstin.strip() and not valid_gstin(b_gstin):
        st.warning("Buyer GSTIN format looks wrong. A GSTIN has 15 characters.")

    c3, c4, c5 = st.columns(3)
    inv_no = c3.text_input("Invoice number", value="INV-0001", key="inv_no")
    inv_date = c4.date_input("Invoice date", value=dt.date.today(), key="inv_date")
    supply = supply_key(c5.radio("Supply type", SUPPLY_OPTIONS, key="inv_supply"))

    with st.expander("Look up GST % from the dataset"):
        ref = pd.DataFrame([{"Item": r["name"], "HSN/SAC": r["hsn_sac"],
                             "Total GST %": fmt_pct(total_gst_pct(r))} for r in RECORDS])
        st.dataframe(ref, hide_index=True, use_container_width=True)
        st.caption("Copy the GST % into the item table below. Check the notification before invoicing.")

    st.markdown("**Items**")
    edited = st.data_editor(
        pd.DataFrame(st.session_state.inv_rows),
        num_rows="dynamic",
        use_container_width=True,
        key="inv_editor",
        column_config={
            "Description": st.column_config.TextColumn("Item description", width="large"),
            "HSN/SAC": st.column_config.TextColumn("HSN/SAC"),
            "Qty": st.column_config.NumberColumn("Qty", min_value=0.0, step=1.0, format="%.2f"),
            "Unit price (Rs)": st.column_config.NumberColumn("Unit price (Rs)", min_value=0.0, format="%.2f"),
            "GST %": st.column_config.NumberColumn("GST %", min_value=0.0, max_value=100.0, format="%.3f"),
        },
    )
    st.session_state.inv_rows = edited.to_dict("records")

    lines = []
    for r in st.session_state.inv_rows:
        desc = _txt(r.get("Description"))
        if not desc:
            continue
        lines.append({
            "description": desc,
            "hsn": _txt(r.get("HSN/SAC")),
            "qty": _num(r.get("Qty"), 1.0),
            "unit_price": _num(r.get("Unit price (Rs)"), 0.0),
            "gst_pct": _num(r.get("GST %"), 0.0),
        })

    if not lines:
        st.info("Add at least one item with a description to see the invoice totals.")
    else:
        try:
            result = compute_invoice(lines, supply)
        except ValueError as err:
            st.error(str(err))
            result = None

        if result:
            st.markdown("**Invoice preview**")
            st.dataframe(pd.DataFrame([{
                "Item": l["description"], "HSN/SAC": l["hsn"], "Qty": l["qty"],
                "Taxable": l["taxable"], "GST %": l["gst_pct"] + "%",
                "Tax": l["tax"], "Total": l["total"],
            } for l in result["lines"]]), hide_index=True, use_container_width=True)

            m1, m2, m3, m4, m5 = st.columns(5)
            m1.metric("Taxable value", f"₹{result['taxable']}")
            m2.metric("CGST", f"₹{result['cgst']}")
            m3.metric("SGST/UTGST", f"₹{result['sgst']}")
            m4.metric("IGST", f"₹{result['igst']}")
            m5.metric("Grand total", f"₹{result['grand_total']}")

            if not s_name.strip() or not b_name.strip():
                st.warning("Enter both seller and buyer names to download the PDF.")
            else:
                seller = {"name": s_name.strip(), "gstin": s_gstin.strip().upper(),
                          "address": s_addr.strip()}
                buyer = {"name": b_name.strip(), "gstin": b_gstin.strip().upper(),
                         "address": b_addr.strip()}
                inv = {"number": inv_no.strip() or "INV-0001", "date": inv_date.isoformat(),
                       "supply": supply}
                pdf_bytes = build_invoice_pdf(seller, buyer, inv, result, DISCLAIMER)
                safe_no = re.sub(r"[^A-Za-z0-9_-]+", "_", inv["number"])
                st.download_button("Download invoice PDF", data=pdf_bytes,
                                   file_name=f"invoice_{safe_no}.pdf", mime="application/pdf",
                                   key="dl_invoice", type="primary")

    st.button("Clear invoice", on_click=clear_invoice, key="inv_clear")


def bulk_page():
    page_header("bulk")
    st.write("Upload a CSV with a column named **item**. Each row is matched to the goods list.")
    sample = pd.DataFrame({"item": ["rice", "cement", "laptop", "fresh milk"]})
    st.download_button("Download sample CSV", data=sample.to_csv(index=False).encode("utf-8-sig"),
                       file_name="bulk_sample.csv", mime="text/csv", key="dl_sample")

    up = st.file_uploader("CSV file", type=["csv"], key="bulk_file")
    if up is None:
        return
    try:
        df = pd.read_csv(up)
    except Exception:
        st.error("Could not read this CSV file.")
        return

    cols = {str(c).strip().lower(): c for c in df.columns}
    if "item" not in cols:
        st.error('The CSV needs a column named "item".')
        return

    items = [str(x).strip() for x in df[cols["item"]].dropna().tolist() if str(x).strip()][:200]
    rows = []
    for item in items:
        exact, _ = search_text(item, RECORDS, "goods")
        if exact:
            rec = exact[0]
            note = f"{len(exact)} matches; first shown" if len(exact) > 1 else ""
            rows.append({"Your item": item, "Matched item": rec["name"], "HSN/SAC": rec["hsn_sac"],
                         "Total GST": fmt_pct(total_gst_pct(rec)), "Status": status_text(rec),
                         "Note": note})
        else:
            rows.append({"Your item": item, "Matched item": "Not found", "HSN/SAC": "",
                         "Total GST": "", "Status": "Not found", "Note": "Try Search for this item"})

    if not rows:
        st.info("The CSV has no items to check.")
        return
    st.dataframe(pd.DataFrame(rows), hide_index=True, use_container_width=True)
    st.download_button("Download results (CSV)", data=to_csv(rows),
                       file_name=f"bulk_results_{TODAY}.csv", mime="text/csv", key="dl_bulk")


def guide_page():
    page_header("guide")
    topic = st.selectbox("Choose a guide", list(GUIDES), key="guide_topic")
    g = GUIDES[topic]
    st.markdown(f"### {g['title']}")
    st.write(g["summary"])
    for i, step in enumerate(g["steps"], 1):
        st.markdown(f"**Step {i}.** {step}")

    st.markdown("#### Videos")
    if not g["videos"]:
        st.info(VIDEO_NOTE)
    for video in g["videos"]:
        st.write(f"**{video['title']}**")
        if video["url"]:
            st.video(video["url"])
        else:
            st.info(VIDEO_NOTE)


def clear_all():
    clear_invoice()
    st.session_state.history = []
    for key in ["query", "scan_typed", "calc_base", "calc_rate", "bulk_file"]:
        st.session_state.pop(key, None)


def privacy_page():
    page_header("privacy")
    st.markdown("""
**What the app keeps**
- Searches and calculations need no personal data. The app never asks for Aadhaar, PAN, bank or card details, contacts, location or ID documents.
- Recent searches are off by default. If you turn them on, they stay in this session's server memory only.
- Invoice details (seller and buyer names, addresses, GSTIN) are used only to build the PDF in your session. They are not saved to disk or to any database.
- Camera and photo scans are read in memory to find a code. The image is not saved by the app.
- Downloads contain only the data you asked for.

**Hosting and vendors**
- The hosting provider is to be confirmed before launch. It may keep standard server and access logs under its own policy.
- The app has no advertising, behavioural analytics or session-replay tools.

**Your rights and complaints**
- Under the Digital Personal Data Protection Act, 2023, you may seek access, correction, erasure, grievance redressal and nomination. The main obligations take effect on 13 May 2027 (G.S.R. 843(E)).
- Grievance contact: [add a working privacy email before launch].

**Status**
This app is designed to align with the DPDP framework. It is not a claim of compliance and is subject to a documented pre-launch review.
""")
    st.divider()
    st.button("Delete all session data now", on_click=clear_all, key="clear_privacy")
    st.caption("Clears recent searches, invoice details, scanner input and bulk uploads for this session.")


def render_sidebar():
    with st.sidebar:
        st.markdown("## ⚠️ Disclaimer")
        with st.expander("Read the full disclaimer"):
            st.write(DISCLAIMER)
        st.markdown("## 📘 About this data")
        st.write(f"**Dataset version:** {META['dataset_version']}")
        st.write(f"**Last updated:** {META['last_updated']}")
        st.caption(META["status"])
        st.caption(f"Source: [{META['source']}]({META['source']})")


ROUTES = {
    "splash": splash,
    "menu": menu_page,
    "search": search_page,
    "browse": browse_page,
    "calc": calc_page,
    "scan": scan_page,
    "invoice": invoice_page,
    "bulk": bulk_page,
    "guide": guide_page,
    "privacy": privacy_page,
}


def main():
    render_sidebar()
    page = st.session_state.page
    ROUTES.get(page, splash)()
    if page not in ("splash", "menu"):
        st.divider()
        st.caption("Informational only. Read the full disclaimer in the sidebar (open with »).")


main()

Writing /content/gst-rate-finder/app.py


In [ ]:
%%writefile /content/gst-rate-finder/tests/test_engine.py
from decimal import Decimal

from gst_engine import (
    NOT_FOUND_MSG,
    calculate,
    load_dataset,
    normalise,
    search_code,
    search_text,
    total_gst_pct,
)

META, RECORDS = load_dataset()


def by_id(record_id):
    return next(r for r in RECORDS if r["id"] == record_id)


def test_dataset_loads_with_unique_ids():
    ids = [r["id"] for r in RECORDS]
    assert len(ids) == len(set(ids))
    assert len(ids) >= 30


def test_prepackaged_rice_is_5_percent_total():
    assert total_gst_pct(by_id("G001")) == Decimal("5")


def test_loose_rice_is_exempt_zero():
    rec = by_id("G002")
    assert rec["treatment"] == "exempt"
    assert total_gst_pct(rec) == Decimal("0")


def test_cement_is_18_percent_total():
    assert total_gst_pct(by_id("G016")) == Decimal("18")


def test_gold_jewellery_is_3_percent_total():
    assert total_gst_pct(by_id("G017")) == Decimal("3")


def test_exact_name_search_returns_rice_records():
    exact, _ = search_text("rice", RECORDS, "goods")
    assert {"G001", "G002"} <= {r["id"] for r in exact}


def test_misspelling_gives_suggestion_not_exact_match():
    exact, suggestions = search_text("ricee", RECORDS, "goods")
    assert exact == []
    assert "G001" in {r["id"] for r in suggestions}


def test_unknown_item_returns_nothing():
    exact, suggestions = search_text("zqxvwk", RECORDS, "goods")
    assert exact == [] and suggestions == []
    assert NOT_FOUND_MSG.startswith("Item not found")


def test_hsn_prefix_search():
    assert {"G001", "G002"} <= {r["id"] for r in search_code("1006", RECORDS, "goods")}


def test_hsn_code_must_be_4_6_or_8_digits():
    try:
        search_code("12", RECORDS, "goods")
    except ValueError:
        return
    raise AssertionError("Expected ValueError for a 2-digit code")


def test_service_name_search_uses_service_kind_only():
    exact, _ = search_text("restaurant", RECORDS, "service")
    assert exact and all(r["kind"] == "service" for r in exact)


def test_normalise_strips_punctuation_and_case():
    assert normalise("  Fresh-Milk!!  ") == "fresh milk"


def test_add_mode_intra_state():
    res = calculate("1000", "18", "intra", "add")
    assert res["total_tax"] == "180.00"
    assert res["cgst"] == "90.00"
    assert res["sgst_utgst"] == "90.00"
    assert res["igst"] == "0.00"
    assert res["final_amount"] == "1180.00"


def test_extract_mode_inter_state():
    res = calculate("1180", "18", "inter", "extract")
    assert res["taxable_value"] == "1000.00"
    assert res["igst"] == "180.00"
    assert res["cgst"] == "0.00"


def test_split_always_adds_up_with_odd_paise():
    res = calculate("100.01", "5", "intra", "add")
    assert abs(float(res["cgst"]) + float(res["sgst_utgst"]) - float(res["total_tax"])) < 0.001


def test_invalid_amount_rejected():
    for bad in ["0", "-5", "abc"]:
        try:
            calculate(bad, "18", "intra", "add")
        except ValueError:
            continue
        raise AssertionError(f"Expected ValueError for amount {bad!r}")

Writing /content/gst-rate-finder/tests/test_engine.py


In [ ]:
%%writefile /content/gst-rate-finder/tests/test_new_features.py
from barcode_engine import ean_valid, is_product_code, lookup_barcode
from gst_engine import load_dataset
from invoice_engine import compute_invoice, valid_gstin

META, RECORDS = load_dataset()


def test_invoice_intra_state_splits_half_and_half():
    res = compute_invoice(
        [{"description": "Test item", "hsn": "1006", "qty": 1, "unit_price": 1000, "gst_pct": 18}],
        "intra")
    assert res["taxable"] == "1000.00"
    assert res["cgst"] == "90.00"
    assert res["sgst"] == "90.00"
    assert res["igst"] == "0.00"
    assert res["grand_total"] == "1180.00"


def test_invoice_inter_state_uses_igst():
    res = compute_invoice(
        [{"description": "Test item", "hsn": "1006", "qty": 2, "unit_price": 500, "gst_pct": 5}],
        "inter")
    assert res["cgst"] == "0.00"
    assert res["igst"] == "50.00"
    assert res["grand_total"] == "1050.00"


def test_invoice_rejects_empty_and_bad_rate():
    try:
        compute_invoice([], "intra")
    except ValueError:
        pass
    else:
        raise AssertionError("Expected error for no items")
    try:
        compute_invoice([{"description": "x", "qty": 1, "unit_price": 1, "gst_pct": 150}], "intra")
    except ValueError:
        return
    raise AssertionError("Expected error for GST above 100")


def test_gstin_format():
    assert valid_gstin("00AAAAA0000A1Z5")
    assert not valid_gstin("00AAAAA0000A1Z")
    assert not valid_gstin("")


def test_ean13_check_digit():
    assert ean_valid("4006381333931")
    assert not ean_valid("4006381333932")


def test_product_code_length_and_digits():
    assert is_product_code("4006381333931")
    assert not is_product_code("ABC123")
    assert not is_product_code("12345")


def test_unlinked_barcode_returns_none():
    assert lookup_barcode("4006381333931", [], RECORDS) is None

Writing /content/gst-rate-finder/tests/test_new_features.py


In [ ]:
%cd /content/gst-rate-finder
!python -m pytest -q

/content/gst-rate-finder
.......................                                                  [100%]
23 passed in 0.07s


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://samuel-lbs-thus-tablet.trycloudflare.com


In [ ]:
import re, pathlib

p = pathlib.Path("/content/gst-rate-finder/app.py")
src = p.read_text(encoding="utf-8")

NEW_CSS = '''CSS = """
<style>
.stApp {
    background:
        radial-gradient(circle at 12% 8%, rgba(255,122,24,0.14), transparent 40%),
        radial-gradient(circle at 88% 18%, rgba(175,0,45,0.16), transparent 40%),
        radial-gradient(circle at 50% 100%, rgba(0,229,255,0.12), transparent 45%),
        #000000;
}
[data-testid="stHeader"] { background: transparent; }
[data-testid="stSidebar"] {
    background: #050508; border-right: 1px solid rgba(255,122,24,0.35);
}
.block-container { padding-top: 2rem; max-width: 1100px; }
h1, h2, h3, h4 { color: #ffffff; }
p, li, label, .stMarkdown { color: #e6e9ef; }

.splash { text-align: center; padding: 7rem 1rem 3rem 1rem; }
.splash h1 {
    font-size: 5.5rem; font-weight: 900; letter-spacing: 4px; margin: 0;
    background: linear-gradient(90deg, #FFD200, #FF7A18, #FF2D55, #B31BFF, #00E5FF);
    -webkit-background-clip: text; background-clip: text; color: transparent;
    filter: drop-shadow(0 0 18px rgba(255,45,85,0.55));
}
.splash p.tag { color: #9aa4b2; letter-spacing: 2px; margin-top: 0.6rem; }

.menu-title {
    text-align: center; font-size: 2.4rem; font-weight: 800;
    background: linear-gradient(90deg, #00E5FF, #B31BFF, #FF2D55);
    -webkit-background-clip: text; background-clip: text; color: transparent;
}

div.stButton > button {
    min-height: 3.4rem; border-radius: 14px; font-size: 1.05rem; font-weight: 700;
    color: #ffffff; background: linear-gradient(135deg, #15151f, #0c0c14);
    border: 1px solid rgba(0,229,255,0.35);
    box-shadow: 0 0 12px rgba(0,229,255,0.12), 0 6px 18px rgba(0,0,0,0.7);
    transition: all 0.2s ease;
}
div.stButton > button:hover {
    transform: translateY(-2px); color: #FFD200; border-color: #FF7A18;
    box-shadow: 0 0 22px rgba(255,122,24,0.5);
}
div.stButton > button[kind="primary"] {
    background: linear-gradient(90deg, #FF7A18, #AF002D); border: none; color: #ffffff;
}
div.stButton > button[kind="primary"]:hover { box-shadow: 0 0 24px rgba(255,45,85,0.6); }

[data-testid="stMetric"] {
    background: linear-gradient(135deg, rgba(255,122,24,0.14), rgba(179,27,255,0.14));
    border: 1px solid rgba(255,122,24,0.4); padding: 0.8rem; border-radius: 14px;
}
[data-testid="stMetricValue"] { color: #FFD200; }
[data-testid="stMetricLabel"] { color: #9be7ff; }

div[data-testid="stVerticalBlockBorderWrapper"] {
    border-radius: 16px !important; border: 1px solid rgba(0,229,255,0.3) !important;
    background: rgba(12,12,20,0.75);
}
[data-testid="stAlert"] { border-radius: 12px; }
</style>
"""'''

src, n = re.subn(r'CSS = """.*?"""', lambda m: NEW_CSS, src, count=1, flags=re.DOTALL)
assert n == 1, "CSS block not found in app.py"

# Use the coloured class for the menu heading and add a tagline on the first page
src = src.replace("<h2 style='text-align:center'>What would you like to do?</h2>",
                  "<h2 class='menu-title'>What would you like to do?</h2>")
src = src.replace('<h1>GST TAXORA</h1></div>',
                  '<h1>GST TAXORA</h1><p class="tag">Find GST rates · Calculate · Invoice · Scan</p></div>')

p.write_text(src, encoding="utf-8")
print("Colourful theme applied to app.py")

Colourful theme applied to app.py


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://bids-tape-unlikely-courier.trycloudflare.com


In [ ]:
%%writefile /content/gst-rate-finder/gstin_engine.py
"""GSTIN format and check-character validation. This does NOT confirm registration."""
from __future__ import annotations

import re

CHARSET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
GSTIN_RE = re.compile(r"^(\d{2})([A-Z]{5}\d{4}[A-Z])([1-9A-Z])Z([0-9A-Z])$")

STATE_CODES = {
    "01": "Jammu and Kashmir", "02": "Himachal Pradesh", "03": "Punjab", "04": "Chandigarh",
    "05": "Uttarakhand", "06": "Haryana", "07": "Delhi", "08": "Rajasthan",
    "09": "Uttar Pradesh", "10": "Bihar", "11": "Sikkim", "12": "Arunachal Pradesh",
    "13": "Nagaland", "14": "Manipur", "15": "Mizoram", "16": "Tripura", "17": "Meghalaya",
    "18": "Assam", "19": "West Bengal", "20": "Jharkhand", "21": "Odisha",
    "22": "Chhattisgarh", "23": "Madhya Pradesh", "24": "Gujarat",
    "26": "Dadra and Nagar Haveli and Daman and Diu", "27": "Maharashtra", "29": "Karnataka",
    "30": "Goa", "31": "Lakshadweep", "32": "Kerala", "33": "Tamil Nadu", "34": "Puducherry",
    "35": "Andaman and Nicobar Islands", "36": "Telangana", "37": "Andhra Pradesh",
    "38": "Ladakh", "97": "Other Territory", "99": "Centre Jurisdiction",
}


def normalise_gstin(raw: str) -> str:
    return re.sub(r"\s+", "", (raw or "")).upper()


def checksum_char(first14: str) -> str:
    total = 0
    for i, ch in enumerate(first14):
        product = CHARSET.index(ch) * (1 if i % 2 == 0 else 2)
        total += product // 36 + product % 36
    return CHARSET[(36 - total % 36) % 36]


def validate_gstin(raw: str) -> dict:
    g = normalise_gstin(raw)
    base = {"gstin": g, "state": "", "valid": False}
    if len(g) != 15:
        return {**base, "message": f"A GSTIN has 15 characters. You entered {len(g)}."}
    m = GSTIN_RE.match(g)
    if not m:
        return {**base, "message": "The format is wrong. Check the letters and digits."}
    state_code = m.group(1)
    if state_code not in STATE_CODES:
        return {**base, "message": f"State code {state_code} is not recognised."}
    if checksum_char(g[:14]) != g[14]:
        return {**base, "message": "The check character does not match, so this GSTIN likely has a typo."}
    return {
        "gstin": g,
        "state": STATE_CODES[state_code],
        "valid": True,
        "message": "Format and check character are valid. This does NOT confirm the number is registered.",
    }


def format_details_text(gstin: str, state: str, d: dict | None) -> str:
    lines = [f"GSTIN: {gstin}", f"State: {state}"]
    if d:
        lines += [
            f"Legal name: {d['legal_name']}",
            f"Trade name: {d['trade_name'] or '-'}",
            f"Address: {d['address'] or '-'}",
            f"Registration date: {d['reg_date'] or '-'}",
            f"Business type: {d['business_type'] or '-'}",
            f"Taxpayer category: {d['category'] or '-'}",
            f"Status (as entered): {d['status']}",
        ]
    lines.append("Note: details entered by a user, not verified by GST TAXORA.")
    return "\n".join(lines)

Writing /content/gst-rate-finder/gstin_engine.py


In [ ]:
%%writefile /content/gst-rate-finder/gstin_pdf.py
"""PDF report for a GSTIN check. Not an official GST certificate."""
from html import escape
from io import BytesIO

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


def build_gstin_report(gstin: str, state: str, d: dict | None, validity: str,
                       disclaimer: str, generated_on: str) -> bytes:
    buf = BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm,
                            topMargin=16 * mm, bottomMargin=16 * mm,
                            title=f"GSTIN report {gstin}")
    ss = getSampleStyleSheet()
    small = ParagraphStyle("small", parent=ss["Normal"], fontSize=9, leading=12)

    story = [
        Paragraph("GSTIN verification report", ss["Title"]),
        Paragraph(f"Generated by GST TAXORA on {escape(generated_on)}. "
                  "This is not an official GST certificate.", small),
        Spacer(1, 8),
    ]
    rows = [
        ("GSTIN", gstin),
        ("State (from GSTIN)", state),
        ("Format and check character", validity),
    ]
    if d:
        rows += [
            ("Legal name", d["legal_name"]),
            ("Trade name", d["trade_name"] or "-"),
            ("Principal place of business", d["address"] or "-"),
            ("Registration date", d["reg_date"] or "-"),
            ("Business type", d["business_type"] or "-"),
            ("Taxpayer category", d["category"] or "-"),
            ("Registration status (as entered)", d["status"]),
            ("Details entered by", f"User, on {d['saved_on']}. Not verified by GST TAXORA."),
        ]
    else:
        rows.append(("Business details", "Not saved. Verify on the official GST portal."))

    table = Table(
        [[Paragraph(f"<b>{escape(k)}</b>", small), Paragraph(escape(str(v)), small)] for k, v in rows],
        colWidths=[60 * mm, 110 * mm],
    )
    table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#F3F3F3")),
    ]))
    story += [table, Spacer(1, 12), Paragraph(escape(disclaimer), small)]
    doc.build(story)
    return buf.getvalue()

Writing /content/gst-rate-finder/gstin_pdf.py


In [ ]:
%%writefile /content/gst-rate-finder/data/news.json
{
  "meta": {
    "last_checked": "2026-11-18",
    "note": "Curated only from the notification texts in the project files. Add an item only after you have checked it on the official source. The app does not fetch live news."
  },
  "items": [
    {"id": "N001", "date": "2026-10-27", "effective": "2026-11-01", "category": "Rate change",
     "title": "Retreaded tractor tyres at 2.5%; other retreaded tyres at 9%",
     "summary": "Notification 02/2026 adds retreaded tractor tyres at 2.5% (Schedule I, S.No 272A), keeps other retreaded or used tyres at 9%, and widens the fertiliser entry to include seaweed-based bio-stimulants.",
     "reference": "Notification 02/2026-Central Tax (Rate), Schedule I S.No 237 and 272A; Schedule II S.No 138",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N002", "date": "2026-10-27", "effective": "2026-11-01", "category": "Rate change",
     "title": "Psyllium seeds (isabgol) added at nil rate",
     "summary": "Notification 03/2026 adds psyllium seeds under heading 1211 90 13 to the nil-rate schedule of notification 10/2025 as S.No 87A.",
     "reference": "Notification 03/2026-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N003", "date": "2026-10-27", "effective": "2026-11-01", "category": "Rate change",
     "title": "Hotel, restaurant, EV transport and delivery service conditions revised",
     "summary": "Notification 06/2026 revises the input tax credit conditions for hotel, restaurant and outdoor catering services. It adds 2.5% entries for passenger and goods transport in electrically operated vehicles, and changes the delivery service entry. It also updates the goods transport agency definition.",
     "reference": "Notification 06/2026-Central Tax (Rate) amending notification 11/2017-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N004", "date": "2026-10-27", "effective": "2026-11-01", "category": "Exemption",
     "title": "New exemptions: SPFO services, toll collection, coffee curing, storage of seeds and grains",
     "summary": "Notification 07/2026 adds nil-rated entries for services by the Seamen's Provident Fund Organisation, toll collection by government bodies, curing of coffee for growers, and storage of cereals, pulses, fruits, vegetables and seeds meant for sowing. It also adds a definition of 'seed'.",
     "reference": "Notification 07/2026-Central Tax (Rate) amending notification 12/2017-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N005", "date": "2026-10-27", "effective": "2026-11-01", "category": "New rule",
     "title": "Reverse charge extended to used cooking oil, plastic and tyre waste, and e-waste",
     "summary": "Notification 04/2026 adds used cooking oil, waste and scrap of plastics, waste and scrap of tyres, and electrical and electronics waste to the list where tax is payable by the registered recipient when the supplier is unregistered.",
     "reference": "Notification 04/2026-Central Tax (Rate) amending notification 4/2017-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N006", "date": "2026-10-27", "effective": "2026-11-01", "category": "New rule",
     "title": "'Nominated agency' definition updated for exports",
     "summary": "Notification 05/2026 changes the definition of 'nominated agency' to refer to entities listed in Appendix 4B of the DGFT Handbook of Procedures, as amended from time to time.",
     "reference": "Notification 05/2026-Central Tax (Rate) amending notification 26/2018-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N007", "date": "2026-04-30", "effective": "2026-05-01", "category": "Rate change",
     "title": "HSN codes restated for some beverages",
     "summary": "Notification 01/2026 changes the HSN entries for certain beverages in Schedule I (2.5%) and Schedule III (20%). The notification changes the codes listed in each entry. Check the entry for your product before invoicing.",
     "reference": "Notification 01/2026-Central Tax (Rate) amending notification 09/2025-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"},
    {"id": "N008", "date": "2025-11-13", "effective": "13 Nov 2025 (some sections); 13 May 2027 (main duties)", "category": "Announcement",
     "title": "DPDP Act: first provisions in force; main duties from May 2027",
     "summary": "Notification G.S.R. 843(E) brings some provisions of the Digital Personal Data Protection Act, 2023 into force. Main obligations for data fiduciaries are scheduled for 13 May 2027. The DPDP Rules, 2025 (G.S.R. 846(E)) also commence in stages.",
     "reference": "G.S.R. 843(E), 13 Nov 2025; DPDP Rules, 2025, G.S.R. 846(E)",
     "source_url": "https://www.meity.gov.in/"},
    {"id": "N009", "date": "2025-09-17", "effective": "2025-09-22", "category": "Rate change",
     "title": "GST rate rationalisation for goods and services",
     "summary": "Central tax rates were revised across many goods and services from 22 September 2025 through notifications 9/2025, 10/2025 and 15/2025. Check the schedule entry for your item.",
     "reference": "Notifications 9/2025, 10/2025 and 15/2025-Central Tax (Rate)",
     "source_url": "https://cbic-gst.gov.in/"}
  ]
}

Writing /content/gst-rate-finder/data/news.json


In [ ]:
%%writefile /content/gst-rate-finder/gstin_pages.py
"""GSTIN search, business details, history, favourites, reports, sharing and news."""
import datetime as dt
import json
from pathlib import Path
from urllib.parse import quote

import streamlit as st

from gst_engine import DISCLAIMER
from gstin_engine import format_details_text, validate_gstin
from gstin_pdf import build_gstin_report

BASE_DIR = Path(__file__).resolve().parent
NEWS_FILE = BASE_DIR / "data" / "news.json"
OFFICIAL_SEARCH_URL = "https://services.gst.gov.in/services/searchtp"
HISTORY_MAX = 15
STATUS_OPTIONS = ["Active", "Suspended", "Cancelled", "Not checked"]


def _init():
    st.session_state.setdefault("gstin_history", [])
    st.session_state.setdefault("gstin_favs", {})
    st.session_state.setdefault("gstin_details", {})
    st.session_state.setdefault("gstin_last_seen", "")


def _go(page):
    st.session_state.page = page


def _open(gstin):
    st.session_state.gstin_input = gstin
    st.session_state.page = "gstin"


def _header(title, icon):
    left, right = st.columns([1.3, 5])
    left.button("← Back to menu", key=f"gback_{title}", on_click=_go, args=("menu",))
    right.markdown(f"## {icon} {title}")


def _remember(gstin, name):
    hist = st.session_state.gstin_history
    hist[:] = [h for h in hist if h["gstin"] != gstin]
    hist.insert(0, {"gstin": gstin, "name": name,
                    "when": dt.datetime.now().strftime("%d %b %Y, %H:%M")})
    del hist[HISTORY_MAX:]


def _toggle_fav(gstin):
    favs = st.session_state.gstin_favs
    if gstin in favs:
        del favs[gstin]
    else:
        d = st.session_state.gstin_details.get(gstin)
        favs[gstin] = {"name": d["legal_name"] if d else "", "details": d}


def _clear_hist():
    st.session_state.gstin_history = []
    st.session_state.gstin_last_seen = ""


def clear_gstin_session():
    st.session_state.gstin_history = []
    st.session_state.gstin_favs = {}
    st.session_state.gstin_details = {}
    st.session_state.gstin_last_seen = ""
    st.session_state.pop("gstin_input", None)


def _details_card(gstin, info, d):
    with st.container(border=True):
        st.subheader(d["legal_name"])
        st.caption(f"GSTIN {gstin} · {info['state']}")
        st.markdown(f"**Trade name:** {d['trade_name'] or '—'}")
        st.markdown(f"**Principal place of business:** {d['address'] or '—'}")
        st.markdown(f"**Registration date:** {d['reg_date'] or '—'}")
        st.markdown(f"**Business type:** {d['business_type'] or '—'}")
        st.markdown(f"**Taxpayer category:** {d['category'] or '—'}")
        st.markdown(f"**Registration status (as entered):** {d['status']}")
        st.caption(f"Entered by you on {d['saved_on']}. Not verified by GST TAXORA.")


def _actions(gstin, info, d):
    is_fav = gstin in st.session_state.gstin_favs
    text = format_details_text(gstin, info["state"], d)
    today = dt.date.today().strftime("%d %b %Y")
    pdf = build_gstin_report(gstin, info["state"], d, info["message"], DISCLAIMER, today)
    c1, c2, c3, c4 = st.columns(4)
    c1.button("⭐ Remove favourite" if is_fav else "☆ Save to favourites",
              key=f"fav_{gstin}", on_click=_toggle_fav, args=(gstin,))
    c2.download_button("Download PDF", data=pdf, file_name=f"GSTIN_{gstin}.pdf",
                       mime="application/pdf", key=f"pdf_{gstin}")
    c3.link_button("Share on WhatsApp", "https://wa.me/?text=" + quote(text), key=f"wa_{gstin}")
    c4.link_button("Share by email",
                   "mailto:?subject=" + quote(f"GSTIN {gstin} details") + "&body=" + quote(text),
                   key=f"mail_{gstin}")
    st.caption("To copy everything, use the copy icon at the top right of the box below.")
    st.code(text, language=None)


def gstin_page():
    _init()
    _header("GSTIN search & verify", "🪪")
    st.text_input("Enter a 15-digit GSTIN", key="gstin_input", max_chars=20,
                  placeholder="e.g. 27AAPFU0939F1ZV")
    raw = st.session_state.get("gstin_input", "") or ""

    if not raw.strip():
        st.caption("Type a GSTIN to check its format and check character. If you have verified the "
                   "details on the official portal, save them here to build a report.")
        st.link_button("Open the official GST portal search ↗", OFFICIAL_SEARCH_URL, key="official_idle")
        return

    info = validate_gstin(raw)
    if not info["valid"]:
        st.error(info["message"])
        return

    gstin = info["gstin"]
    store = st.session_state.gstin_details
    d = store.get(gstin)
    if st.session_state.gstin_last_seen != gstin:
        _remember(gstin, d["legal_name"] if d else "No details saved")
        st.session_state.gstin_last_seen = gstin

    st.success(info["message"])
    st.write(f"**State:** {info['state']} (from the first two digits)")
    st.warning("This app does not contact the GST portal. Registration status is shown only "
               "if you enter it below after checking the official portal yourself.")
    st.link_button("Verify on the official GST portal ↗", OFFICIAL_SEARCH_URL, key=f"official_{gstin}")

    with st.expander("Enter details you verified on the official portal", expanded=d is None):
        with st.form(f"form_{gstin}"):
            legal = st.text_input("Legal name *")
            trade = st.text_input("Trade name (optional)")
            addr = st.text_area("Principal place of business (address)", height=80)
            reg_date = st.text_input("Registration date (DD/MM/YYYY)")
            btype = st.text_input("Business type (e.g. Proprietorship, Private Limited)")
            category = st.text_input("Taxpayer category (e.g. Regular, Composition)")
            status = st.selectbox("Registration status", STATUS_OPTIONS)
            submitted = st.form_submit_button("Save for this session")
        if submitted:
            if not legal.strip():
                st.warning("Enter the legal name to save.")
            else:
                d = {
                    "legal_name": legal.strip(), "trade_name": trade.strip(),
                    "address": addr.strip(), "reg_date": reg_date.strip(),
                    "business_type": btype.strip(), "category": category.strip(),
                    "status": status, "saved_on": dt.date.today().strftime("%d %b %Y"),
                }
                store[gstin] = d
                if gstin in st.session_state.gstin_favs:
                    st.session_state.gstin_favs[gstin] = {"name": d["legal_name"], "details": d}
                _remember(gstin, d["legal_name"])
                st.success("Saved for this session.")

    if d is None:
        st.info("No business details saved yet for this GSTIN.")
        st.code(gstin, language=None)
        return

    _details_card(gstin, info, d)
    _actions(gstin, info, d)


def history_page():
    _init()
    _header("Search history", "🕘")
    hist = st.session_state.gstin_history
    st.caption("Kept only in this session. Use Search again to open a GSTIN without typing it.")
    if not hist:
        st.info("No GSTIN searches yet in this session.")
        return
    for i, h in enumerate(hist):
        with st.container(border=True):
            c1, c2 = st.columns([4, 1.4])
            c1.markdown(f"**{h['gstin']}**")
            c1.caption(f"{h['name']} · {h['when']}")
            c2.button("Search again", key=f"hagain_{i}_{h['gstin']}", on_click=_open, args=(h["gstin"],))
    st.button("Clear history", key="hclear", on_click=_clear_hist)


def favs_page():
    _init()
    _header("Favourites", "⭐")
    favs = st.session_state.gstin_favs
    if not favs:
        st.info("No favourites yet. Open a GSTIN and tap Save to favourites.")
        return
    for gstin, entry in list(favs.items()):
        d = entry.get("details") or st.session_state.gstin_details.get(gstin)
        with st.container(border=True):
            st.markdown(f"**{gstin}**")
            st.caption(entry.get("name") or "No details saved")
            if d:
                st.write(f"Trade name: {d['trade_name'] or '—'} · Status (as entered): {d['status']}")
                st.write(f"Address: {d['address'] or '—'}")
            c1, c2 = st.columns(2)
            c1.button("Open", key=f"fopen_{gstin}", on_click=_open, args=(gstin,))
            c2.button("Remove", key=f"frem_{gstin}", on_click=_toggle_fav, args=(gstin,))


@st.cache_data
def _load_news():
    with open(NEWS_FILE, encoding="utf-8") as fh:
        return json.load(fh)


def news_page():
    _init()
    _header("GST news & updates", "📰")
    data = _load_news()
    items = sorted(data["items"], key=lambda x: x["date"], reverse=True)
    st.caption(f"Curated from the notifications in the project files, last checked "
               f"{data['meta']['last_checked']}. This app does not fetch live news, so check "
               "CBIC for the latest.")
    cats = ["All"] + sorted({i["category"] for i in items})
    choice = st.selectbox("Show", cats, key="news_cat")
    for item in [i for i in items if choice == "All" or i["category"] == choice]:
        with st.container(border=True):
            st.markdown(f"**{item['title']}**")
            st.caption(f"{item['category']} · Notified {item['date']} · Effective {item['effective']}")
            st.write(item["summary"])
            st.caption(f"Reference: {item['reference']}")
            st.link_button("Check on the official site ↗", item["source_url"], key=f"news_{item['id']}")


GSTIN_MENU = [
    ("gstin", "🪪", "GSTIN search & verify", "Check format, view and save business details"),
    ("ghist", "🕘", "Search history", "Recent GSTIN searches, search again in one click"),
    ("gfav", "⭐", "Favourites", "Saved GST numbers for quick access"),
    ("gnews", "📰", "GST news & updates", "Rate changes, notifications and new rules"),
]

GSTIN_ROUTES = {
    "gstin": gstin_page,
    "ghist": history_page,
    "gfav": favs_page,
    "gnews": news_page,
}

Writing /content/gst-rate-finder/gstin_pages.py


In [ ]:
import pathlib

p = pathlib.Path("/content/gst-rate-finder/app.py")
src = p.read_text(encoding="utf-8")


def swap(old, new):
    global src
    assert src.count(old) == 1, f"Expected exactly one match for: {old[:70]!r}"
    src = src.replace(old, new)


swap("from invoice_pdf import build_invoice_pdf\n",
     "from invoice_pdf import build_invoice_pdf\n"
     "from gstin_pages import GSTIN_MENU, GSTIN_ROUTES, clear_gstin_session\n")
swap('    ("privacy", "🔒", "Privacy & data", "See what is kept and delete your session data"),\n',
     '    *GSTIN_MENU,\n'
     '    ("privacy", "🔒", "Privacy & data", "See what is kept and delete your session data"),\n')
swap('    "privacy": privacy_page,\n}',
     '    "privacy": privacy_page,\n    **GSTIN_ROUTES,\n}')
swap("def clear_all():\n    clear_invoice()\n",
     "def clear_all():\n    clear_invoice()\n    clear_gstin_session()\n")

p.write_text(src, encoding="utf-8")
print("app.py updated with GSTIN, history, favourites and news pages")

app.py updated with GSTIN, history, favourites and news pages


In [ ]:
%cd /content/gst-rate-finder
!python -m pytest -q

/content/gst-rate-finder
.......................                                                  [100%]
23 passed in 0.04s


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://large-catalog-chapel-desert.trycloudflare.com


In [ ]:
import pathlib, re

p = pathlib.Path("/content/gst-rate-finder/app.py")
src = p.read_text(encoding="utf-8")

CALM_CSS = '''CSS = """
<style>
.stApp {
    background:
        radial-gradient(circle at 0% 0%, rgba(255,122,24,0.08), transparent 35%),
        radial-gradient(circle at 100% 0%, rgba(175,0,45,0.08), transparent 35%),
        #000000;
}
[data-testid="stHeader"] { background: transparent; }
[data-testid="stSidebar"] {
    background: #050508; border-right: 1px solid rgba(255,122,24,0.25);
}
.block-container { padding-top: 1.5rem; max-width: 1040px; }
h1, h2, h3 { color: #ffffff; font-weight: 700; }
p, li, label { color: #d5d9e0; }

.splash { text-align: center; padding: 7rem 1rem 3rem 1rem; }
.splash h1 {
    font-size: 5rem; font-weight: 900; letter-spacing: 4px; margin: 0;
    background: linear-gradient(90deg, #FF7A18, #FF2D55, #B31BFF);
    -webkit-background-clip: text; background-clip: text; color: transparent;
}
.splash p.tag { color: #8b93a1; letter-spacing: 2px; margin-top: 0.6rem; }

.menu-title { text-align: center; font-size: 2rem; font-weight: 700; color: #ffffff; margin-bottom: 0.2rem; }
.menu-sub { text-align: center; color: #8b93a1; margin-bottom: 1.6rem; }
.menu-section {
    color: #FF7A18; font-size: 0.78rem; font-weight: 700; letter-spacing: 2px;
    text-transform: uppercase; margin: 1.6rem 0 0.6rem 0;
}

/* Calm buttons: no glow at rest, a soft accent on hover */
div.stButton > button {
    min-height: 3rem; border-radius: 12px; font-size: 1rem; font-weight: 600;
    color: #e6e9ef; background: #14141c;
    border: 1px solid #262633; box-shadow: none;
    transition: border-color 0.15s ease, background 0.15s ease;
}
div.stButton > button:hover {
    color: #ffffff; border-color: #FF7A18; background: #1a1a24;
}
div.stButton > button[kind="primary"] {
    background: linear-gradient(90deg, #FF7A18, #AF002D); border: none; color: #ffffff;
}

/* One accent per card: subtitle text sits under the button */
.menu-desc { color: #8b93a1; font-size: 0.85rem; margin: 0.25rem 0 0.9rem 0.2rem; }

[data-testid="stMetric"] {
    background: #101018; border: 1px solid #262633; padding: 0.8rem; border-radius: 12px;
}
[data-testid="stMetricValue"] { color: #FFD200; }
[data-testid="stMetricLabel"] { color: #9aa4b2; }

div[data-testid="stVerticalBlockBorderWrapper"] {
    border-radius: 12px !important; border: 1px solid #22222c !important;
    background: #0e0e15;
}
[data-testid="stAlert"] { border-radius: 10px; }
</style>
"""'''

src, n = re.subn(r'CSS = """.*?"""', lambda m: CALM_CSS, src, count=1, flags=re.DOTALL)
assert n == 1, "CSS block not found"

# New menu: grouped sections, one heading, short subtitles under each button
NEW_MENU = '''def menu_page():
    st.markdown("<div class='menu-title'>What would you like to do?</div>", unsafe_allow_html=True)
    st.markdown("<div class='menu-sub'>Pick one tool to get started.</div>", unsafe_allow_html=True)
    sections = [
        ("GST rates and tax", ["search", "browse", "calc"]),
        ("Products and billing", ["scan", "invoice", "bulk"]),
        ("GSTIN and business", ["gstin", "ghist", "gfav", "gnews"]),
        ("Help and privacy", ["guide", "privacy"]),
    ]
    by_key = {m[0]: m for m in MENU}
    for heading, keys in sections:
        st.markdown(f"<div class='menu-section'>{heading}</div>", unsafe_allow_html=True)
        cols = st.columns(2)
        for i, key in enumerate(keys):
            _, icon, title, desc = by_key[key]
            with cols[i % 2]:
                st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                          on_click=go, args=(key,))
                st.markdown(f"<div class='menu-desc'>{desc}</div>", unsafe_allow_html=True)
'''

src, n = re.subn(r"def menu_page\(\):.*?(?=\n\ndef )", lambda m: NEW_MENU.rstrip("\n"),
                 src, count=1, flags=re.DOTALL)
assert n == 1, "menu_page function not found"

# Calmer splash: drop the tagline colour clash
src = src.replace('<p class="tag">Find GST rates · Calculate · Invoice · Scan</p>',
                  '<p class="tag">GST rates · Calculator · Invoice · GSTIN</p>')

p.write_text(src, encoding="utf-8")
print("Menu layout and calmer styling applied")

Menu layout and calmer styling applied


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://flush-councils-ferrari-coating.trycloudflare.com


In [ ]:
import pathlib, re

p = pathlib.Path("/content/gst-rate-finder/app.py")
src = p.read_text(encoding="utf-8")

THEME_BLOCK = r'''THEMES = {
    "Dark": {
        "bg": "#000000", "panel": "#0b0b12", "card": "#14141c", "text": "#ffffff",
        "muted": "#9aa4b2", "border": "#2a2a38", "accent": "#FF7A18", "accent2": "#FF2D55",
        "metric": "#FFD200",
    },
    "Light": {
        "bg": "#f6f7fb", "panel": "#ffffff", "card": "#ffffff", "text": "#111827",
        "muted": "#4b5563", "border": "#d6d9e0", "accent": "#E8590C", "accent2": "#C2255C",
        "metric": "#B45309",
    },
}

CSS_TEMPLATE = """
<style>
.stApp { background: $bg; }
[data-testid="stHeader"] { background: transparent; }
[data-testid="stSidebar"] { background: $panel; border-right: 1px solid $border; }
.block-container { padding-top: 1.5rem; max-width: 1040px; }
h1, h2, h3, h4 { color: $text !important; font-weight: 700; }
p, li, label, .stMarkdown, .stCaption, [data-testid="stCaptionContainer"] { color: $muted; }
.stMarkdown p, .stMarkdown li, label { color: $text; }

.splash { text-align: center; padding: 7rem 1rem 3rem 1rem; }
.splash h1 {
    font-size: 5rem; font-weight: 900; letter-spacing: 4px; margin: 0;
    background: linear-gradient(90deg, #FFD200, #FF7A18, #FF2D55, #B31BFF, #00E5FF);
    -webkit-background-clip: text; background-clip: text; color: transparent;
}
.splash p.tag { color: $muted; letter-spacing: 2px; margin-top: 0.6rem; }

.menu-title { text-align: center; font-size: 2rem; font-weight: 700; color: $text; margin-bottom: 0.2rem; }
.menu-sub { text-align: center; color: $muted; margin-bottom: 1.6rem; }
.menu-section {
    font-size: 0.8rem; font-weight: 800; letter-spacing: 2px; text-transform: uppercase;
    margin: 1.6rem 0 0.6rem 0;
}
.menu-desc { color: $muted; font-size: 0.85rem; margin: 0.25rem 0 0.9rem 0.2rem; }

div.stButton > button {
    min-height: 3rem; border-radius: 12px; font-size: 1rem; font-weight: 600;
    color: $text; background: $card; border: 1px solid $border; box-shadow: none;
    transition: border-color 0.15s ease, background 0.15s ease;
}
div.stButton > button:hover { color: $text; border-color: $accent; background: $card; }
div.stButton > button[kind="primary"] {
    background: linear-gradient(90deg, $accent, $accent2); border: none; color: #ffffff;
}

[data-testid="stMetric"] {
    background: $card; border: 1px solid $border; padding: 0.8rem; border-radius: 12px;
}
[data-testid="stMetricValue"] { color: $metric; }
[data-testid="stMetricLabel"] { color: $muted; }

div[data-testid="stVerticalBlockBorderWrapper"] {
    border-radius: 12px !important; border: 1px solid $border !important; background: $panel;
}
[data-testid="stAlert"] { border-radius: 10px; }
</style>
"""


def build_css(theme: str) -> str:
    tokens = THEMES.get(theme, THEMES["Dark"])
    css = CSS_TEMPLATE
    # Longest keys first so "accent2" is replaced before "accent"
    for key in sorted(tokens, key=len, reverse=True):
        css = css.replace("$" + key, tokens[key])
    return css
'''

# 1. Replace the old hard-coded CSS block and its st.markdown call
src, n = re.subn(
    r'CSS = """.*?"""\nst\.markdown\(CSS, unsafe_allow_html=True\)',
    lambda m: THEME_BLOCK + '\nst.markdown(build_css(st.session_state.get("theme", "Dark")), unsafe_allow_html=True)',
    src, count=1, flags=re.DOTALL)
assert n == 1, "Old CSS block not found in app.py"

# 2. Add the Colour theme switch at the top of the sidebar
assert src.count("    with st.sidebar:\n") == 1, "Sidebar block not found"
src = src.replace(
    "    with st.sidebar:\n",
    "    with st.sidebar:\n"
    '        st.radio("Colour theme", ["Dark", "Light"], horizontal=True, key="theme")\n'
    "        st.divider()\n",
)

# 3. Menu with a different accent colour per section
NEW_MENU = '''def menu_page():
    st.markdown("<div class='menu-title'>What would you like to do?</div>", unsafe_allow_html=True)
    st.markdown("<div class='menu-sub'>Pick one tool to get started.</div>", unsafe_allow_html=True)
    sections = [
        ("GST rates and tax", "#FF7A18", ["search", "browse", "calc"]),
        ("Products and billing", "#00E5FF", ["scan", "invoice", "bulk"]),
        ("GSTIN and business", "#B31BFF", ["gstin", "ghist", "gfav", "gnews"]),
        ("Help and privacy", "#FFD200", ["guide", "privacy"]),
    ]
    by_key = {m[0]: m for m in MENU}
    for heading, colour, keys in sections:
        st.markdown(f"<div class='menu-section' style='color:{colour}'>{heading}</div>",
                    unsafe_allow_html=True)
        cols = st.columns(2)
        for i, key in enumerate(keys):
            _, icon, title, desc = by_key[key]
            with cols[i % 2]:
                st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                          on_click=go, args=(key,))
                st.markdown(f"<div class='menu-desc'>{desc}</div>", unsafe_allow_html=True)
'''
src, n = re.subn(r"def menu_page\(\):.*?(?=\n\ndef )", lambda m: NEW_MENU.rstrip("\n"),
                 src, count=1, flags=re.DOTALL)
assert n == 1, "menu_page function not found"

p.write_text(src, encoding="utf-8")
print("Colour theme switch added (sidebar) and more colours applied")

Colour theme switch added (sidebar) and more colours applied


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://initially-youth-referrals-closed.trycloudflare.com


In [ ]:
import pathlib

p = pathlib.Path("/content/gst-rate-finder/app.py")
src = p.read_text(encoding="utf-8")

NEW_THEME = r'''from string import Template

THEMES = {
    "Dark": {
        "bg": "radial-gradient(circle at 0% 0%, rgba(255,122,24,0.20), transparent 35%), radial-gradient(circle at 100% 15%, rgba(179,27,255,0.18), transparent 40%), radial-gradient(circle at 50% 100%, rgba(0,229,255,0.12), transparent 45%), #000000",
        "text": "#ffffff", "muted": "#a7b0bd", "panel": "#08080d", "card": "#13131c",
        "input": "#101018", "border": "#2c2c3a", "accent": "#FF7A18", "accent2": "#FF2D55",
        "metric": "#FFD200", "metric_bg": "linear-gradient(135deg, rgba(255,122,24,0.18), rgba(179,27,255,0.18))",
        "glow": "rgba(255,122,24,0.40)",
    },
    "Light": {
        "bg": "linear-gradient(135deg, #fff7ed 0%, #fdf2f8 50%, #ecfeff 100%)",
        "text": "#111827", "muted": "#4b5563", "panel": "#ffffff", "card": "#ffffff",
        "input": "#ffffff", "border": "#d4d8e0", "accent": "#E8590C", "accent2": "#C2255C",
        "metric": "#B45309", "metric_bg": "linear-gradient(135deg, #fff1e6, #fce7f3)",
        "glow": "rgba(232,89,12,0.25)",
    },
}

CSS_TEMPLATE = Template("""<style>
.stApp { background: $bg !important; }
[data-testid="stHeader"] { background: transparent; }
[data-testid="stSidebar"] { background: $panel; border-right: 1px solid $border; }
.block-container { padding-top: 1.2rem; max-width: 1040px; }

h1, h2, h3, h4 { color: $text !important; font-weight: 700; }
p, li, label, [data-testid="stCaptionContainer"], .stCaption { color: $muted !important; }
.stMarkdown p, .stMarkdown li { color: $text !important; }
[data-testid="stRadio"] label p, [data-testid="stCheckbox"] label p,
[data-testid="stWidgetLabel"] p { color: $text !important; }

/* Inputs, selects, text areas */
[data-baseweb="input"], [data-baseweb="base-input"], [data-baseweb="select"] > div,
[data-baseweb="textarea"], div[data-baseweb="input"] > div {
    background: $input !important; border-color: $border !important;
}
input, textarea, [data-baseweb="select"] span { color: $text !important; }

/* Tabs */
[data-baseweb="tab"] { color: $muted !important; }
[data-baseweb="tab"][aria-selected="true"] { color: $accent !important; }
[data-baseweb="tab-highlight"] { background: $accent !important; }
[data-baseweb="tab-border"] { background: $border !important; }

/* Expanders, code blocks, alerts */
[data-testid="stExpander"] details { background: $card; border: 1px solid $border; border-radius: 12px; }
[data-testid="stExpander"] summary p, [data-testid="stExpander"] summary span { color: $text !important; }
pre, code { background: $card !important; color: $text !important; border: 1px solid $border; }
[data-testid="stAlert"] { border-radius: 10px; }

/* Title splash */
.splash { text-align: center; padding: 6rem 1rem 3rem 1rem; }
.splash h1 {
    font-size: 5rem; font-weight: 900; letter-spacing: 4px; margin: 0;
    background: linear-gradient(90deg, #FFD200, #FF7A18, #FF2D55, #B31BFF, #00E5FF);
    -webkit-background-clip: text; background-clip: text; color: transparent !important;
}
.splash p.tag { color: $muted !important; letter-spacing: 2px; margin-top: 0.6rem; }

/* Menu */
.menu-title { text-align: center; font-size: 2rem; font-weight: 800; color: $text; margin-bottom: 0.2rem; }
.menu-sub { text-align: center; color: $muted !important; margin-bottom: 1.4rem; }
.menu-section { font-size: 0.8rem; font-weight: 800; letter-spacing: 2px; text-transform: uppercase;
    margin: 1.6rem 0 0.6rem 0; }
.menu-desc { color: $muted !important; font-size: 0.85rem; margin: 0.25rem 0 0.9rem 0.2rem; }

div.stButton > button {
    min-height: 3rem; border-radius: 12px; font-size: 1rem; font-weight: 600;
    color: $text !important; background: $card; border: 1px solid $border;
    box-shadow: 0 2px 10px rgba(0,0,0,0.12);
    transition: all 0.18s ease;
}
div.stButton > button:hover {
    color: $text !important; border-color: $accent; background: $card;
    box-shadow: 0 0 18px $glow; transform: translateY(-1px);
}
div.stButton > button[kind="primary"] {
    background: linear-gradient(90deg, $accent, $accent2) !important; border: none; color: #ffffff !important;
}

/* Metrics and bordered cards */
[data-testid="stMetric"] {
    background: $metric_bg; border: 1px solid $border; padding: 0.8rem; border-radius: 14px;
}
[data-testid="stMetricValue"] { color: $metric !important; }
[data-testid="stMetricLabel"] p { color: $muted !important; }
div[data-testid="stVerticalBlockBorderWrapper"] {
    border-radius: 14px !important; border: 1px solid $border !important; background: $panel;
}
</style>""")


def build_css(theme: str) -> str:
    tokens = THEMES.get(theme, THEMES["Dark"])
    return CSS_TEMPLATE.safe_substitute(tokens)


def apply_theme():
    choice = st.radio("Appearance", ["🌙 Dark", "☀️ Light"], horizontal=True,
                      key="theme_choice", label_visibility="collapsed")
    theme = "Light" if choice.startswith("☀") else "Dark"
    st.markdown(build_css(theme), unsafe_allow_html=True)
    return theme
'''

start = src.index("THEMES = {") if "THEMES = {" in src else src.index('CSS = """')
end_marker = 'st.markdown(build_css(st.session_state.get("theme", "Dark")), unsafe_allow_html=True)'
if end_marker in src:
    end = src.index(end_marker) + len(end_marker)
else:
    end = src.index("st.markdown(CSS, unsafe_allow_html=True)") + len("st.markdown(CSS, unsafe_allow_html=True)")
src = src[:start] + NEW_THEME + src[end:]

# Remove the old sidebar switch if present
old_sidebar = '        st.radio("Colour theme", ["Dark", "Light"], horizontal=True, key="theme")\n        st.divider()\n'
src = src.replace(old_sidebar, "")

# Apply the theme at the top of every page
assert src.count("def main():\n    render_sidebar()\n") == 1, "main() not found as expected"
src = src.replace("def main():\n    render_sidebar()\n",
                  "def main():\n    render_sidebar()\n    apply_theme()\n")

# Menu with coloured cards (left bar per card, section colours)
NEW_MENU = '''def menu_page():
    st.markdown("<div class='menu-title'>What would you like to do?</div>", unsafe_allow_html=True)
    st.markdown("<div class='menu-sub'>Pick one tool to get started.</div>", unsafe_allow_html=True)
    sections = [
        ("GST rates and tax", "#FF7A18", ["search", "browse", "calc"]),
        ("Products and billing", "#00E5FF", ["scan", "invoice", "bulk"]),
        ("GSTIN and business", "#B31BFF", ["gstin", "ghist", "gfav", "gnews"]),
        ("Help and privacy", "#FFD200", ["guide", "privacy"]),
    ]
    by_key = {m[0]: m for m in MENU}
    card_css = "".join(
        f".st-key-menu_{key} button {{ border-left: 5px solid {colour} !important; }}"
        for _, colour, keys in sections for key in keys
    )
    st.markdown(f"<style>{card_css}</style>", unsafe_allow_html=True)
    for heading, colour, keys in sections:
        st.markdown(f"<div class='menu-section' style='color:{colour}'>{heading}</div>",
                    unsafe_allow_html=True)
        cols = st.columns(2)
        for i, key in enumerate(keys):
            _, icon, title, desc = by_key[key]
            with cols[i % 2]:
                st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                          on_click=go, args=(key,))
                st.markdown(f"<div class='menu-desc'>{desc}</div>", unsafe_allow_html=True)
'''
src, n = re.subn(r"def menu_page\(\):.*?(?=\n\ndef )", lambda m: NEW_MENU.rstrip("\n"),
                 src, count=1, flags=re.DOTALL) if False else (src, 0)
import re
src, n = re.subn(r"def menu_page\(\):.*?(?=\n\ndef )", lambda m: NEW_MENU.rstrip("\n"),
                 src, count=1, flags=re.DOTALL)
assert n == 1, "menu_page function not found"

p.write_text(src, encoding="utf-8")
print("Theme switch moved to the top of every page, with colourful styling applied")

Theme switch moved to the top of every page, with colourful styling applied


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://wearing-organ-gary-arabic.trycloudflare.com


In [ ]:
import pathlib

p = pathlib.Path("/content/gst-rate-finder/app.py")
src = p.read_text(encoding="utf-8")

# 1. Remove the call that draws the switch and applies the theme
src = src.replace("    apply_theme()\n", "    st.markdown(build_css(\"Dark\"), unsafe_allow_html=True)\n", 1)

# 2. Remove the apply_theme() function so it can't be called again
start = src.find("def apply_theme():")
if start != -1:
    end = src.find("\n\n\n", start)
    end = len(src) if end == -1 else end
    src = src[:start] + src[end:].lstrip("\n")

p.write_text(src, encoding="utf-8")
print("Dark/Light switch removed. The app now uses the Dark theme.")

Dark/Light switch removed. The app now uses the Dark theme.


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://poll-payday-prices-instructor.trycloudflare.com


In [ ]:
import pathlib, re, json

BASE = pathlib.Path("/content/gst-rate-finder")
app = BASE / "app.py"
src = app.read_text(encoding="utf-8")

# 1. Home button on every feature page: add it to page_header
old_header = '''def page_header(key):
    left, right = st.columns([1.3, 5])
    left.button("← Back to menu", key=f"back_{key}", on_click=go, args=("menu",))
    right.markdown(f"## {MENU_ICON[key]} {MENU_TITLE[key]}")'''
new_header = '''def page_header(key):
    home, back, title = st.columns([1, 1.3, 4])
    home.button("🏠 Home", key=f"home_{key}", on_click=go, args=("splash",))
    back.button("← Back to menu", key=f"back_{key}", on_click=go, args=("menu",))
    title.markdown(f"## {MENU_ICON[key]} {MENU_TITLE[key]}")'''
assert src.count(old_header) == 1, "page_header not found as expected"
src = src.replace(old_header, new_header)

# 2. Bigger, clearer menu: one column of large buttons, full width
src, n = re.subn(
    r"def menu_page\(\):.*?(?=\n\ndef )",
    '''def menu_page():
    st.markdown("<div class='menu-title'>What would you like to do?</div>", unsafe_allow_html=True)
    st.markdown("<div class='menu-sub'>Pick one tool to get started.</div>", unsafe_allow_html=True)
    sections = [
        ("GST rates and tax", "#FF7A18", ["search", "browse", "calc"]),
        ("Products and billing", "#00E5FF", ["scan", "invoice", "bulk"]),
        ("GSTIN and business", "#B31BFF", ["gstin", "ghist", "gfav", "gnews"]),
        ("Help and privacy", "#FFD200", ["guide", "privacy"]),
    ]
    by_key = {m[0]: m for m in MENU}
    card_css = "".join(
        f".st-key-menu_{key} button {{ border-left: 6px solid {colour} !important; "
        f"min-height: 4.6rem !important; font-size: 1.25rem !important; }}"
        for _, colour, keys in sections for key in keys
    )
    st.markdown(f"<style>{card_css}</style>", unsafe_allow_html=True)
    for heading, colour, keys in sections:
        st.markdown(f"<div class='menu-section' style='color:{colour}'>{heading}</div>",
                    unsafe_allow_html=True)
        for key in keys:
            _, icon, title, desc = by_key[key]
            st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                      on_click=go, args=(key,))
            st.markdown(f"<div class='menu-desc'>{desc}</div>", unsafe_allow_html=True)''',
    src, count=1, flags=re.DOTALL)
assert n == 1, "menu_page not found"

app.write_text(src, encoding="utf-8")

# 3. News: each item gets its own link. Store the links in the data file.
news_path = BASE / "data" / "news.json"
news = json.loads(news_path.read_text(encoding="utf-8"))
daily_sources = [
    ("TaxSutra GST", "https://www.taxsutra.com/gst"),
    ("SAG Infotech GST updates", "https://blog.saginfotech.com/gst-latest-updates"),
    ("TaxAJ daily compliance updates", "https://www.taxaj.com/daily-tax-compliance-updates"),
]
for i, item in enumerate(news["items"]):
    if item.get("category") == "Announcement" and "DPDP" in item["title"]:
        item["source_url"] = "https://www.meity.gov.in/"
        item["source_name"] = "MeitY (official)"
    else:
        name, url = daily_sources[i % len(daily_sources)]
        item["source_url"] = url
        item["source_name"] = name
news["meta"]["note"] = ("Curated from the notification texts in the project files. Each item links to a "
                        "source for daily updates. Check the link yourself before relying on it.")
news_path.write_text(json.dumps(news, indent=2, ensure_ascii=False), encoding="utf-8")

# 4. Use the per-item source name on the button
gp = BASE / "gstin_pages.py"
g = gp.read_text(encoding="utf-8")
old_btn = 'st.link_button("Check on the official site ↗", item["source_url"], key=f"news_{item[\'id\']}")'
assert g.count(old_btn) == 1, "news button line not found"
g = g.replace(old_btn,
              'st.link_button(f"Read more on {item.get(\'source_name\', \'the official site\')} ↗", '
              'item["source_url"], key=f"news_{item[\'id\']}")')
gp.write_text(g, encoding="utf-8")

print("Home button added, menu enlarged, and each news item now links to its own source.")

Home button added, menu enlarged, and each news item now links to its own source.


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://limitations-suited-broken-this.trycloudflare.com


In [ ]:
import pathlib, re

app = pathlib.Path("/content/gst-rate-finder/app.py")
src = app.read_text(encoding="utf-8")

NEW_MENU = '''def menu_page():
    home, _ = st.columns([1, 4])
    home.button("🏠 Home", key="home_menu", on_click=go, args=("splash",))
    st.markdown("<div class='menu-sub'>Pick one tool to get started.</div>", unsafe_allow_html=True)
    sections = [
        ("GST rates and tax", "#FF7A18", ["search", "browse", "calc"]),
        ("Products and billing", "#00E5FF", ["scan", "invoice", "bulk"]),
        ("GSTIN and business", "#B31BFF", ["gstin", "ghist", "gfav", "gnews"]),
        ("Help and privacy", "#FFD200", ["guide", "privacy"]),
    ]
    by_key = {m[0]: m for m in MENU}
    card_css = "".join(
        f".st-key-menu_{key} button {{ border-left: 6px solid {colour} !important; "
        f"min-height: 5rem !important; }}"
        f".st-key-menu_{key} button, .st-key-menu_{key} button p, .st-key-menu_{key} button span "
        f"{{ font-size: 1.45rem !important; font-weight: 700 !important; }}"
        for _, colour, keys in sections for key in keys
    )
    st.markdown(f"<style>{card_css}"
                ".menu-desc { font-size: 1.05rem !important; margin-bottom: 1.1rem !important; }"
                ".menu-section { font-size: 1rem !important; }</style>", unsafe_allow_html=True)
    for heading, colour, keys in sections:
        st.markdown(f"<div class='menu-section' style='color:{colour}'>{heading}</div>",
                    unsafe_allow_html=True)
        for key in keys:
            _, icon, title, desc = by_key[key]
            st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                      on_click=go, args=(key,))
            st.markdown(f"<div class='menu-desc'>{desc}</div>", unsafe_allow_html=True)'''

src, n = re.subn(r"def menu_page\(\):.*?(?=\n\ndef )", lambda m: NEW_MENU, src, count=1, flags=re.DOTALL)
assert n == 1, "menu_page function not found"
app.write_text(src, encoding="utf-8")
print("Menu updated: Home button added, heading removed, larger options.")

Menu updated: Home button added, heading removed, larger options.


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://brings-patio-resources-congress.trycloudflare.com


In [ ]:
import pathlib, re

app = pathlib.Path("/content/gst-rate-finder/app.py")
src = app.read_text(encoding="utf-8")

NEW_MENU = '''def menu_page():
    home, _ = st.columns([1, 4])
    home.button("🏠 Home", key="home_menu", on_click=go, args=("splash",))
    sections = [
        ("GST rates and tax", "#FF7A18", ["search", "browse", "calc"]),
        ("Products and billing", "#00E5FF", ["scan", "invoice", "bulk"]),
        ("GSTIN and business", "#B31BFF", ["gstin", "ghist", "gfav", "gnews"]),
        ("Help and privacy", "#FFD200", ["guide", "privacy"]),
    ]
    by_key = {m[0]: m for m in MENU}
    card_css = "".join(
        f".st-key-menu_{key} button {{ border-left: 6px solid {colour} !important; "
        f"min-height: 3.6rem !important; }}"
        f".st-key-menu_{key} button, .st-key-menu_{key} button p, .st-key-menu_{key} button span "
        f"{{ font-size: 1.25rem !important; font-weight: 700 !important; }}"
        for _, colour, keys in sections for key in keys
    )
    st.markdown(
        f"<style>{card_css}"
        ".menu-desc { font-size: 0.98rem !important; margin: 0.15rem 0 0.7rem 0.2rem !important; }"
        ".menu-section { font-size: 0.95rem !important; margin: 1.1rem 0 0.4rem 0 !important; }"
        "div[data-testid='stVerticalBlock'] > div { gap: 0.35rem; }"
        "</style>",
        unsafe_allow_html=True,
    )
    for heading, colour, keys in sections:
        st.markdown(f"<div class='menu-section' style='color:{colour}'>{heading}</div>",
                    unsafe_allow_html=True)
        for key in keys:
            _, icon, title, desc = by_key[key]
            st.button(f"{icon}  {title}", key=f"menu_{key}", use_container_width=True,
                      on_click=go, args=(key,))
            st.markdown(f"<div class='menu-desc'>{desc}</div>", unsafe_allow_html=True)'''

src, n = re.subn(r"def menu_page\(\):.*?(?=\n\ndef )", lambda m: NEW_MENU, src, count=1, flags=re.DOTALL)
assert n == 1, "menu_page function not found"
app.write_text(src, encoding="utf-8")
print("Menu spacing tidied, subtitle removed, button size kept.")

Menu spacing tidied, subtitle removed, button size kept.


In [ ]:
import time, re, subprocess

subprocess.run("pkill -f streamlit || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(2)

subprocess.Popen(
    "streamlit run /content/gst-rate-finder/app.py --server.port 8501 --server.headless true "
    "> /content/streamlit.log 2>&1", shell=True)
time.sleep(8)
!curl -s -o /dev/null -w "Streamlit HTTP status: %{http_code}\n" http://localhost:8501

subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > /content/tunnel.log 2>&1", shell=True)
time.sleep(12)
log = open("/content/tunnel.log").read()
urls = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
print("Open this link:", urls[0] if urls else "Not found yet. Run this cell again in a few seconds.")

Streamlit HTTP status: 200
Open this link: https://ddr-listening-firewall-gif.trycloudflare.com


In [ ]:
import shutil, pathlib, zipfile
from google.colab import files

src = pathlib.Path("/content/gst-rate-finder")

# Remove caches so the zip stays clean
for cache in list(src.rglob("__pycache__")) + list(src.rglob(".pytest_cache")):
    shutil.rmtree(cache, ignore_errors=True)

# Add a .gitignore so caches and secrets stay out of any future repo
(src / ".gitignore").write_text(
    ".venv/\n__pycache__/\n.pytest_cache/\n.env\n.streamlit/secrets.toml\n",
    encoding="utf-8",
)

zip_path = shutil.make_archive("/content/GST_TAXORA", "zip", root_dir="/content", base_dir="gst-rate-finder")
print("Created:", zip_path)
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        print("  ", name)

files.download(zip_path)

Created: /content/GST_TAXORA.zip
   gst-rate-finder/
   gst-rate-finder/data/
   gst-rate-finder/tests/
   gst-rate-finder/gstin_pdf.py
   gst-rate-finder/gstin_pages.py
   gst-rate-finder/gst_engine.py
   gst-rate-finder/gstin_engine.py
   gst-rate-finder/.gitignore
   gst-rate-finder/invoice_engine.py
   gst-rate-finder/barcode_engine.py
   gst-rate-finder/requirements.txt
   gst-rate-finder/invoice_pdf.py
   gst-rate-finder/guides.py
   gst-rate-finder/app.py
   gst-rate-finder/data/gst_rates.json
   gst-rate-finder/data/news.json
   gst-rate-finder/data/barcodes.json
   gst-rate-finder/tests/test_new_features.py
   gst-rate-finder/tests/test_engine.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pathlib, shutil, zipfile
from google.colab import files

src = pathlib.Path("/content/gst-rate-finder")
required = [
    "app.py", "gst_engine.py", "barcode_engine.py", "gstin_engine.py", "gstin_pages.py",
    "gstin_pdf.py", "guides.py", "invoice_engine.py", "invoice_pdf.py", "requirements.txt",
    "data/gst_rates.json", "data/barcodes.json", "data/news.json",
    "tests/test_engine.py", "tests/test_new_features.py", "tests/test_gstin.py",
]

missing = [f for f in required if not (src / f).exists()]
if missing:
    raise SystemExit("Missing files, fix before zipping:\n  " + "\n  ".join(missing))
print("All required files found.")

for cache in list(src.rglob("__pycache__")) + list(src.rglob(".pytest_cache")):
    shutil.rmtree(cache, ignore_errors=True)

(src / ".gitignore").write_text(
    ".venv/\n__pycache__/\n.pytest_cache/\n.env\n.streamlit/secrets.toml\n", encoding="utf-8")

zip_path = "/content/GST_TAXORA_flat.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in required + [".gitignore"]:
        z.write(src / f, arcname=f)

with zipfile.ZipFile(zip_path) as z:
    print("\nZip contents:")
    for name in z.namelist():
        print("  ", name)

files.download(zip_path)

SystemExit: Missing files, fix before zipping:
  tests/test_gstin.py

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
%%writefile /content/gst-rate-finder/tests/test_gstin.py
import json
from pathlib import Path

from gstin_engine import checksum_char, format_details_text, validate_gstin
from gstin_pdf import build_gstin_report

NEWS = Path("/content/gst-rate-finder/data/news.json")


def test_known_valid_gstin_passes_checksum():
    res = validate_gstin("27AAPFU0939F1ZV")
    assert res["valid"] is True
    assert res["state"] == "Maharashtra"


def test_wrong_check_character_fails():
    res = validate_gstin("27AAPFU0939F1ZW")
    assert res["valid"] is False
    assert "typo" in res["message"]


def test_wrong_length_fails():
    res = validate_gstin("27AAPFU0939F1Z")
    assert res["valid"] is False
    assert "15 characters" in res["message"]


def test_spaces_and_lowercase_are_normalised():
    assert validate_gstin(" 27aapfu0939f1zv ")["valid"] is True


def test_bad_format_fails():
    assert validate_gstin("ABAAPFU0939F1ZV")["valid"] is False


def test_checksum_char_matches_known_example():
    assert checksum_char("27AAPFU0939F1Z") == "V"


def test_details_text_without_details_still_works():
    text = format_details_text("27AAPFU0939F1ZV", "Maharashtra", None)
    assert text.startswith("GSTIN: 27AAPFU0939F1ZV")


def test_pdf_report_is_pdf_bytes():
    pdf = build_gstin_report("27AAPFU0939F1ZV", "Maharashtra", None,
                             "Format valid", "Test disclaimer", "18 Nov 2026")
    assert pdf.startswith(b"%PDF")


def test_news_file_has_required_fields():
    data = json.loads(NEWS.read_text(encoding="utf-8"))
    required = {"id", "date", "effective", "category", "title", "summary", "reference", "source_url"}
    assert data["items"]
    for item in data["items"]:
        assert required <= item.keys()

Writing /content/gst-rate-finder/tests/test_gstin.py


In [ ]:
%cd /content/gst-rate-finder
!python -m pytest -q

/content/gst-rate-finder
................................                                         [100%]
32 passed in 0.22s


In [ ]:
import pathlib, shutil, zipfile
from google.colab import files

src = pathlib.Path("/content/gst-rate-finder")
required = [
    "app.py", "gst_engine.py", "barcode_engine.py", "gstin_engine.py", "gstin_pages.py",
    "gstin_pdf.py", "guides.py", "invoice_engine.py", "invoice_pdf.py", "requirements.txt",
    "data/gst_rates.json", "data/barcodes.json", "data/news.json",
    "tests/test_engine.py", "tests/test_new_features.py", "tests/test_gstin.py",
]

missing = [f for f in required if not (src / f).exists()]
if missing:
    raise SystemExit("Missing files, fix before zipping:\n  " + "\n  ".join(missing))
print("All required files found.")

for cache in list(src.rglob("__pycache__")) + list(src.rglob(".pytest_cache")):
    shutil.rmtree(cache, ignore_errors=True)

(src / ".gitignore").write_text(
    ".venv/\n__pycache__/\n.pytest_cache/\n.env\n.streamlit/secrets.toml\n", encoding="utf-8")

zip_path = "/content/GST_TAXORA_flat.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in required + [".gitignore"]:
        z.write(src / f, arcname=f)

with zipfile.ZipFile(zip_path) as z:
    print("\nZip contents:")
    for name in z.namelist():
        print("  ", name)

files.download(zip_path)

All required files found.

Zip contents:
   app.py
   gst_engine.py
   barcode_engine.py
   gstin_engine.py
   gstin_pages.py
   gstin_pdf.py
   guides.py
   invoice_engine.py
   invoice_pdf.py
   requirements.txt
   data/gst_rates.json
   data/barcodes.json
   data/news.json
   tests/test_engine.py
   tests/test_new_features.py
   tests/test_gstin.py
   .gitignore


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pathlib, shutil, zipfile
from google.colab import files

base = pathlib.Path("/content/gst-rate-finder")

FINDER = '''def _find_data(name):
    """Look in data/ first, then the top level, so either layout works."""
    for candidate in (BASE_DIR / "data" / name, BASE_DIR / name):
        if candidate.exists():
            return candidate
    return BASE_DIR / "data" / name
'''

def patch(filename, old, new):
    path = base / filename
    text = path.read_text(encoding="utf-8")
    assert text.count(old) == 1, f"{filename}: expected one match for {old!r}"
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("Patched", filename)

# gst_engine.py
patch("gst_engine.py",
      'DATA_FILE = BASE_DIR / "data" / "gst_rates.json"\n',
      FINDER + '\nDATA_FILE = _find_data("gst_rates.json")\n')

# barcode_engine.py
patch("barcode_engine.py",
      'BARCODE_FILE = BASE_DIR / "data" / "barcodes.json"\n',
      FINDER + '\nBARCODE_FILE = _find_data("barcodes.json")\n')

# gstin_pages.py
patch("gstin_pages.py",
      'NEWS_FILE = BASE_DIR / "data" / "news.json"\n',
      FINDER + '\nNEWS_FILE = _find_data("news.json")\n')

# Run the tests to confirm nothing broke
!cd /content/gst-rate-finder && python -m pytest -q

# Build a clean flat zip
src = base
required = [
    "app.py", "gst_engine.py", "barcode_engine.py", "gstin_engine.py", "gstin_pages.py",
    "gstin_pdf.py", "guides.py", "invoice_engine.py", "invoice_pdf.py", "requirements.txt",
    "data/gst_rates.json", "data/barcodes.json", "data/news.json",
    "tests/test_engine.py", "tests/test_new_features.py", "tests/test_gstin.py",
]
missing = [f for f in required if not (src / f).exists()]
if missing:
    raise SystemExit("Missing files:\n  " + "\n  ".join(missing))

for cache in list(src.rglob("__pycache__")) + list(src.rglob(".pytest_cache")):
    shutil.rmtree(cache, ignore_errors=True)
(src / ".gitignore").write_text(
    ".venv/\n__pycache__/\n.pytest_cache/\n.env\n.streamlit/secrets.toml\n", encoding="utf-8")

zip_path = "/content/GST_TAXORA_final.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in required + [".gitignore"]:
        z.write(src / f, arcname=f)

with zipfile.ZipFile(zip_path) as z:
    print("\nZip contents:")
    for name in z.namelist():
        print("  ", name)

files.download(zip_path)

Patched gst_engine.py
Patched barcode_engine.py
Patched gstin_pages.py
................................                                         [100%]
32 passed in 0.23s

Zip contents:
   app.py
   gst_engine.py
   barcode_engine.py
   gstin_engine.py
   gstin_pages.py
   gstin_pdf.py
   guides.py
   invoice_engine.py
   invoice_pdf.py
   requirements.txt
   data/gst_rates.json
   data/barcodes.json
   data/news.json
   tests/test_engine.py
   tests/test_new_features.py
   tests/test_gstin.py
   .gitignore


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>